In [1]:
# ============================================================
# 031_pdf_inbox_processor
# ============================================================
#
# Overview
# ----------------
# Processes PDFs from a configured inbox folder,
# extracts and repairs metadata (title, authors, DOI, arXiv ID),
# deduplicates against existing Notion records,
# creates Papers entries with status=INBOX,
# generates one-slide visual summaries,
# uploads artifacts to Google Drive,
# updates Notion with Drive links,
# and moves files to processed/failed folders.
#
# Designed for daily runs with strict idempotency,
# partial-failure tolerance, and safe re-execution.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - PDF files in configured inbox directory (from config)
#   - run_id, logger, config, state helpers (from 028)
#   - Notion wrappers: create_paper_inbox, find_duplicate_*,
#     update_paper_links (from 029)
#
# Outputs:
#   - Notion Papers records (status=INBOX) via 029 wrappers
#   - One-slide artifacts generated via IR → Gemini (PNG)
#   - PDFs and slides uploaded to Google Drive (shareable links)
#   - Drive links written back to Notion Paper records
#   - PDFs moved to processed/ or failed/ subdirectories
#   - Daily run summary (counts, failures, duplicates)
#   - Slack-friendly summary text output
#
# Structure
# ----------------
# Cell 01: Imports and environment validation
# Cell 02: Folder setup and path resolution
# Cell 03: Text normalization and dedup key utilities
# Cell 04: PDF metadata extraction + LLM repair fallback
# Cell 05: Notion property introspection and wrapper probing
# Cell 06: Deduplication checker with fallback logic
# Cell 07: Slide Spec (IR) generation + Gemini slide rendering
# Cell 08: Notion record creation adapter
# Cell 09: Single PDF processing pipeline
# Cell 10: Main processing loop and file orchestration
# Cell 11: Run summary generation and reporting
# Cell 12: Daily summary output and Slack snippet
#
# Notes
# ----------------
# - Assumes 028_config_and_state and 029_notion_clients_and_io
#   are already executed
# - Does NOT initialize logging, run_id, or load env vars
#   (owned by 028)
# - Does NOT construct raw Notion JSON or HTTP calls
#   (uses 029 wrappers only)
# - Implements strict deduplication: DOI → arXiv → normalized title
# - Fully rerunnable: dedup_key prevents duplicate creation
# - Partial failures are tolerated (slide / Drive failures
#   do not block Notion creation unless configured)
# - Slide generation uses explicit IR to reduce hallucination
# - Google Drive auth is assumed external; this notebook
#   consumes an initialized drive_service
# - Files are moved atomically with error capture and logging
# - Defensive coding throughout: wrapper availability,
#   config keys, and state access are always checked


In [2]:
# ============================================================
# Cell 01 — Imports and defensive bootstrap (NO infra re-impl)
# ============================================================
# Overview:
#   Import dependencies and defensively bind runtime objects provided by 028/029.
#   This cell MUST NOT load env vars or initialize clients (owned by 028/029).
#
# Inputs / Outputs:
#   Inputs: (optional) globals from 028_config_and_state and 029_notion_clients_and_io
#   Outputs: run_id/logger/config/get_state/update_state bound (with safe fallbacks)
#
# Notes:
#   - No env loading here (028 owns env/secrets)
#   - No Notion client init here (029 owns Notion side effects)
#   - Be defensive: accept missing globals and fall back to minimal defaults

from __future__ import annotations

import os
import json
import re
import shutil
from pathlib import Path
from datetime import datetime, date
from typing import Dict, List, Optional, Tuple, Any
# ============================================================
# Google Drive Authentication (standalone, run once)
# ============================================================

from pathlib import Path
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow

# ----------------------------
# Config
# ----------------------------
SCOPES = ["https://www.googleapis.com/auth/drive.file"]

TOKEN_PATH = Path("token.json")          # notebook と同じ階層
CLIENT_SECRET_PATH = Path("client_secret.json")  # ← 実ファイル名に合わせること
# ----------------------------
# Helpers
# ----------------------------
def _load_creds_from_token(token_path: Path):
    creds = Credentials.from_authorized_user_file(str(token_path), SCOPES)

    if creds.valid:
        return creds

    if creds.expired and creds.refresh_token:
        try:
            creds.refresh(Request())
            token_path.write_text(creds.to_json(), encoding="utf-8")
            print("🔄 token.json refreshed")
            return creds
        except Exception as e:
            print("⚠️ token refresh failed:", repr(e))
            return None

    return None


def _run_oauth_and_save_token(client_secret_path: Path, token_path: Path):
    if not client_secret_path.exists():
        raise FileNotFoundError(
            f"{client_secret_path.resolve()} not found.\n"
            "Place your OAuth Desktop client secret as client_secret.json next to the notebook."
        )

    flow = InstalledAppFlow.from_client_secrets_file(
        str(client_secret_path),
        SCOPES,
    )
    creds = flow.run_local_server(port=0)
    token_path.write_text(creds.to_json(), encoding="utf-8")
    print(f"💾 token.json regenerated: {token_path.resolve()}")
    return creds


# ----------------------------
# Main
# ----------------------------
creds = None

# 1) Try existing token.json
if TOKEN_PATH.exists():
    creds = _load_creds_from_token(TOKEN_PATH)

# 2) Re-OAuth if needed
if creds is None:
    print("🔐 Google Drive re-authentication required")
    creds = _run_oauth_and_save_token(CLIENT_SECRET_PATH, TOKEN_PATH)

# 3) Build Drive service
drive_service = build("drive", "v3", credentials=creds)

print("✅ Google Drive service initialized and ready")

# PDF libs (use whichever is installed/available downstream)
try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None

try:
    import pdfplumber
except Exception:
    pdfplumber = None

try:
    import PyPDF2
except Exception:
    PyPDF2 = None
# Overview:
#   Ensure 028 and 029 are executed in THIS kernel namespace.
# Notes:
#   - Uses %run -i so variables/functions are injected into current globals()

from pathlib import Path

HERE = Path.cwd()

# If your notebooks are in the same directory, use direct filenames.
nb028 = HERE / "028_config_and_state.ipynb"
nb029 = HERE / "029_notion_clients_and_io.ipynb"

# If they live elsewhere, adjust, e.g. HERE / "notebooks" / "028_config_and_state.ipynb"
assert nb028.exists(), f"Cannot find: {nb028}"
assert nb029.exists(), f"Cannot find: {nb029}"

get_ipython().run_line_magic("run", f"-i {nb028}")
get_ipython().run_line_magic("run", f"-i {nb029}")

print("✅ 028 and 029 executed in current kernel")
print("create_paper_inbox callable:", callable(globals().get("create_paper_inbox")))
print("create_paper callable:", callable(globals().get("create_paper")))
# ------------------------------------------------------------
# Defensive binding of execution context from 028
# ------------------------------------------------------------
def _noop_logger():
    class _L:
        def info(self, *a, **k): print(*a)
        def warning(self, *a, **k): print(*a)
        def error(self, *a, **k): print(*a)
        def debug(self, *a, **k): pass
    return _L()

# run_id
run_id = globals().get("run_id") or f"run_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"

# logger
logger = globals().get("logger") or _noop_logger()

# config: accept common aliases; default to minimal structure
config = (
    globals().get("config")
    or globals().get("CONFIG")
    or globals().get("cfg")
    or globals().get("settings")
    or globals().get("runtime_config")
    or {}
)

# state helpers
get_state = globals().get("get_state")
update_state = globals().get("update_state")

def _fallback_get_state(key: str, default=None):
    return default

def _fallback_update_state(key: str, value):
    return None

if not callable(get_state):
    get_state = _fallback_get_state
if not callable(update_state):
    update_state = _fallback_update_state

# ------------------------------------------------------------
# Paths (defensive defaults)
# ------------------------------------------------------------
paths_cfg = config.get("paths", {}) if isinstance(config, dict) else {}
download_dir = Path(paths_cfg.get("download_dir", "data/downloads"))
artifacts_dir = Path(paths_cfg.get("artifacts_dir", "artifacts"))
processed_dir = download_dir / "processed"
failed_dir = download_dir / "failed"
dupe_dir = processed_dir / "duplicates"

for d in [download_dir, artifacts_dir, processed_dir, failed_dir, dupe_dir]:
    d.mkdir(parents=True, exist_ok=True)

today_str = date.today().isoformat()

logger.info(f"[{run_id}] Cell 01 ready")
logger.info(f"[{run_id}] download_dir={download_dir}")
logger.info(f"[{run_id}] artifacts_dir={artifacts_dir}")

# ------------------------------------------------------------
# Defensive binding of Notion wrappers from 029 (optional here)
# ------------------------------------------------------------
create_paper_inbox = globals().get("create_paper_inbox")
create_paper = globals().get("create_paper")
find_duplicate_by_doi = globals().get("find_duplicate_by_doi")
find_duplicate_by_arxiv_id = globals().get("find_duplicate_by_arxiv_id")
find_duplicate_by_title = globals().get("find_duplicate_by_title")

logger.debug(f"[{run_id}] notion wrappers present: "
             f"create_paper_inbox={callable(create_paper_inbox)}, "
             f"create_paper={callable(create_paper)}, "
             f"find_duplicate_by_doi={callable(find_duplicate_by_doi)}, "
             f"find_duplicate_by_arxiv_id={callable(find_duplicate_by_arxiv_id)}, "
             f"find_duplicate_by_title={callable(find_duplicate_by_title)}")


🔄 token.json refreshed
✅ Google Drive service initialized and ready
✓ Cell 01: Imports and dependencies loaded
✓ Cell 02: Environment bootstrap completed
  - NOTION_TOKEN: ntn_38...***
  - NOTION_VERSION: 2025-09-03
  - NOTION_LIT_DB_ID: set
  - NOTION_EVENTS_DB_ID: set
  - NOTION_MONITORING_TARGETS_DB_ID: set
  - NOTION_MONITORING_QUEUE_DB_ID: set
✓ Cell 03: Loaded configuration from config.yaml
  - Drive folder_id:      (set)
✓ Cell 03: Configuration validated (source: config.yaml)
  - Pipeline cadence:     daily
  - Max runtime:          30 min
  - Lookback (daily):     7 days
  - Lookback (tasks):     14 days
  - Lookback (projects):  30 days
  - Max items per query:  100
  - Logging level:        INFO
✓ Cell 04: Run context initialized
  - Run ID:           3b30808f-8492-4d6d-a373-340713fc1a63
  - Execution start:  2026-02-01T00:31:42.272695+00:00
  - Run date:         2026-02-01
  - Timezone:         UTC
  - Cadence:          daily
  - Lookback windows:
    - daily_notes : 7 days

In [3]:
# ============================================================
# Cell 02 — Folder setup and path resolution (defensive)
# ============================================================
# Overview:
#   Resolve inbox/processed/failed and slide artifact paths using config (defensive)
#   and create required directories.
#
# Inputs / Outputs:
#   Inputs: config (optional), logger/run_id, update_state (optional)
#   Outputs: inbox_path, processed_path, failed_path, slides_path (Path objects)
#
# Notes:
#   - NO direct key access like config['download_directory']
#   - Prefer config['paths']['download_dir'] and config['paths']['artifacts_dir']
#   - Safe to rerun (mkdir exist_ok=True)
#   - Persist paths via update_state if available

from pathlib import Path

def _cfg_get_path(config_obj: Any, *keys: str, default: str) -> str:
    """
    Safely get nested config like config['paths']['download_dir'].
    keys: sequence of nested keys.
    """
    cur = config_obj if isinstance(config_obj, dict) else {}
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k)
        if cur is None:
            return default
    return cur if isinstance(cur, str) and cur.strip() else default

# Prefer the architecture keys: config.paths.*
inbox_raw = _cfg_get_path(config, "paths", "download_dir", default="data/downloads")
artifacts_raw = _cfg_get_path(config, "paths", "artifacts_dir", default="artifacts")

inbox_path = Path(inbox_raw).expanduser().resolve()
artifacts_path = Path(artifacts_raw).expanduser().resolve()

# Ensure base directories
inbox_path.mkdir(parents=True, exist_ok=True)
artifacts_path.mkdir(parents=True, exist_ok=True)

if not inbox_path.is_dir():
    raise ValueError(f"Download directory is not a directory: {inbox_path}")
if not artifacts_path.is_dir():
    raise ValueError(f"Artifacts directory is not a directory: {artifacts_path}")

# Subdirectories
processed_path = inbox_path / "processed"
failed_path = inbox_path / "failed"
dupe_path = processed_path / "duplicates"
slides_path = artifacts_path / "slides"

for d in [processed_path, failed_path, dupe_path, slides_path]:
    d.mkdir(parents=True, exist_ok=True)

logger.info(f"[{run_id}] Inbox path: {inbox_path}")
logger.info(f"[{run_id}] Processed path: {processed_path}")
logger.info(f"[{run_id}] Failed path: {failed_path}")
logger.info(f"[{run_id}] Duplicates path: {dupe_path}")
logger.info(f"[{run_id}] Slides path: {slides_path}")
# Sanity check: warn if inbox_path looks different from expected project layout
try:
    if "notebooks" in str(Path.cwd()) and "notebooks" not in str(inbox_path):
        logger.warning(f"[{run_id}] inbox_path may be unexpected (cwd under notebooks but inbox_path is outside): {inbox_path}")
except Exception:
    pass

# Extra visibility: how many PDFs are currently seen in inbox?
try:
    pdf_count = len(list(inbox_path.glob("*.pdf"))) + len(list(inbox_path.glob("*.PDF")))
    logger.info(f"[{run_id}] Inbox PDF count (pre-run): {pdf_count}")
except Exception:
    pass


# Persist for downstream cells (use update_state if available)
try:
    update_state("paths", {
        "inbox_path": str(inbox_path),
        "processed_path": str(processed_path),
        "failed_path": str(failed_path),
        "dupe_path": str(dupe_path),
        "slides_path": str(slides_path),
    })
except Exception:
    # If update_state isn't available, it's fine; keep variables in memory
    pass


2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Inbox path: /Users/yuetoya/Desktop/researchOS100-private/notebooks/data/downloads
2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Processed path: /Users/yuetoya/Desktop/researchOS100-private/notebooks/data/downloads/processed
2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Failed path: /Users/yuetoya/Desktop/researchOS100-private/notebooks/data/downloads/failed
2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Duplicates path: /Users/yuetoya/Desktop/researchOS100-private/notebooks/data/downloads/processed/duplicates
2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Slides path: /Users/yuetoya/Desktop/researchOS100-private/notebooks/a

In [4]:
# ============================================================
# Cell 03 — Text normalization and dedup key utilities
# ============================================================
# Overview:
#   Provides text normalization functions for deduplication:
#   - normalize_text: lowercases, strips whitespace, removes punctuation
#   - normalize_title: specialized title normalization (removes common prefixes)
#   - make_dedup_key: generates stable hash key from normalized text
#   Used by Cell 06 (dedup checker) and Cell 04 (metadata extraction).
#
# Inputs / Outputs:
#   Inputs: raw strings (titles, DOIs, arXiv IDs)
#   Outputs: normalized strings, dedup keys (hex digest)
#
# Notes:
#   - Normalization is aggressive to maximize dedup recall
#   - Dedup key uses SHA-256 truncated to 16 chars for readability
#   - Title normalization removes common academic prefixes ("A ", "The ", etc.)
#   - All functions are pure (no side effects, no state access)
#   - Safe for Unicode input (NFD normalization)

import hashlib
import unicodedata
import re

def normalize_text(text: Optional[str]) -> str:
    """
    Normalize arbitrary text for deduplication comparison.
    
    Steps:
      1. Return empty string if None/empty
      2. Unicode NFD normalization
      3. Lowercase
      4. Strip leading/trailing whitespace
      5. Replace multiple spaces with single space
      6. Remove common punctuation
    
    Args:
        text: Input string (may be None)
    
    Returns:
        Normalized string (empty if input was None/empty)
    """
    if not text:
        return ""
    
    # Unicode normalization (NFD = decomposed form)
    normalized = unicodedata.normalize('NFD', text)
    
    # Lowercase and strip
    normalized = normalized.lower().strip()
    
    # Collapse multiple spaces
    normalized = re.sub(r'\s+', ' ', normalized)
    
    # Remove common punctuation (keep hyphens, underscores for IDs)
    normalized = re.sub(r'[.,;:!?()\[\]{}"\'\\/]', '', normalized)
    
    return normalized


def normalize_title(title: Optional[str]) -> str:
    """
    Normalize paper title for deduplication.
    
    Applies normalize_text, then removes common academic prefixes:
      - "a ", "an ", "the "
      - "on ", "towards ", "toward "
    
    Args:
        title: Raw paper title (may be None)
    
    Returns:
        Normalized title string
    """
    normalized = normalize_text(title)
    
    if not normalized:
        return ""
    
    # Remove common prefixes (order matters: longest first)
    prefixes = [
        'towards ', 'toward ', 'the ', 'an ', 'a ', 'on '
    ]
    
    for prefix in prefixes:
        if normalized.startswith(prefix):
            normalized = normalized[len(prefix):]
            break  # Remove only first matching prefix
    
    return normalized.strip()


def make_dedup_key(*parts: Optional[str]) -> str:
    """
    Generate stable dedup key from 1+ parts.
    Backward compatible:
      - make_dedup_key("text") もOK
      - make_dedup_key("paper", source_uid) もOK（029互換）
    """
    cleaned = []
    for p in parts:
        s = normalize_text(p) if p is not None else ""
        if s:
            cleaned.append(s)

    if not cleaned:
        return ""

    blob = "|".join(cleaned)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:16]



# --- Self-test ---
logger.info(f"[{run_id}] Cell 03: Text normalization utilities loaded")

# Test normalize_text
test_text = "  A Sample   Title: With Punctuation!  "
test_normalized = normalize_text(test_text)
logger.debug(f"normalize_text('{test_text}') -> '{test_normalized}'")
assert test_normalized == "a sample title with punctuation", "normalize_text failed"

# Test normalize_title
test_title = "Towards a Better Understanding of Deep Learning"
test_title_norm = normalize_title(test_title)
logger.debug(f"normalize_title('{test_title}') -> '{test_title_norm}'")
assert not test_title_norm.startswith('towards'), "normalize_title prefix removal failed"

# Test make_dedup_key
test_key = make_dedup_key("Sample Text")
logger.debug(f"make_dedup_key('Sample Text') -> '{test_key}'")
assert len(test_key) == 16, "make_dedup_key length check failed"
assert test_key == make_dedup_key("sample text"), "make_dedup_key case-insensitivity failed"

# Test empty input handling
assert normalize_text(None) == "", "normalize_text(None) should return empty"
assert normalize_text("") == "", "normalize_text('') should return empty"
assert make_dedup_key(None) == "", "make_dedup_key(None) should return empty"
assert make_dedup_key("") == "", "make_dedup_key('') should return empty"

logger.info(f"[{run_id}] Cell 03: Self-test passed")


2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Cell 03: Text normalization utilities loaded
2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Cell 03: Self-test passed


In [5]:
# ============================================================
# Cell 04 — PDF metadata extraction (DOI, arXiv, title, authors) + OpenAI fallback (JSON)
# ============================================================
# Overview:
#   1) Fast local extraction (PyPDF2 + regex)
#   2) Improved local "title/authors" inference from first ~2 pages text
#   3) If still suspicious, call OpenAI to "repair" metadata (JSON-only).
#
# Notes:
#   - OpenAI is NOT always called. Only when heuristics say "metadata is bad".
#   - If openai_client is missing / OpenAI call fails, we continue with local metadata.
#   - Adds fields: year (optional), authors_year (optional), llm_used, llm_error

import re
import json
from datetime import datetime
from typing import Dict, Any, List, Optional

import PyPDF2

# Prefer PyMuPDF text extraction for reliability (optional)
try:
    import fitz  # PyMuPDF
    _HAS_FITZ = True
except Exception:
    _HAS_FITZ = False

# ----------------------------
# Config knobs
# ----------------------------
LLM_REPAIR_ENABLED = True
OPENAI_MODEL = globals().get("OPENAI_MODEL", "gpt-4.1-mini")
OPENAI_TEMPERATURE = float(globals().get("OPENAI_TEMPERATURE", 0.2))

# How much text to send to OpenAI
LLM_MAX_CHARS = 18000           # keep it compact (fast & cheap)
LLM_PAGES_FOR_TEXT = 2          # first N pages

# Local heuristics: how many pages to use for title/authors inference
LOCAL_PAGES_FOR_INFERENCE = 2

# ----------------------------
# Utilities: suspicious title detection
# ----------------------------
_BAD_TITLE_SUBSTRINGS = [
    "authors listed", "acknowledg", "table of contents", "contents",
    "microsoft word", "working paper", "preprint", "draft",
    "copyright", "all rights reserved", "editorial", "supplementary",
]

def _is_suspicious_title(title: str) -> bool:
    t = (title or "").strip()
    if not t:
        return True
    tl = t.lower()

    for s in _BAD_TITLE_SUBSTRINGS:
        if s in tl:
            return True

    # title is basically a DOI-ish filename
    if re.fullmatch(r"[0-9a-z\.\-_]{10,}\.pdf", tl):
        return True
    if re.search(r"\b10\.\d{4,}/", tl):
        return True

    # too short / too long
    if len(t) < 8 or len(t) > 200:
        return True

    # too many weird symbols
    sym_ratio = sum(1 for ch in t if not (ch.isalnum() or ch.isspace())) / max(len(t), 1)
    if sym_ratio > 0.25:
        return True

    # looks like placeholder
    if tl in {"my title", "untitled", "title"}:
        return True

    return False

# ----------------------------
# Text extraction helper (first N pages)
# ----------------------------
def _extract_first_pages_text(pdf_path: "Path", pages: int = 2) -> str:
    # 1) PyMuPDF preferred
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path)
            chunks = []
            for i in range(min(pages, len(doc))):
                txt = doc[i].get_text("text") or ""
                if txt.strip():
                    chunks.append(txt)
            doc.close()
            return "\n\n".join(chunks).strip()
        except Exception as e:
            logger.warning(f"[{run_id}] fitz text extraction failed; falling back to PyPDF2. err={e}")

    # 2) PyPDF2 fallback
    try:
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            chunks = []
            for i in range(min(pages, len(reader.pages))):
                try:
                    txt = reader.pages[i].extract_text() or ""
                    if txt.strip():
                        chunks.append(txt)
                except Exception:
                    pass
            return "\n\n".join(chunks).strip()
    except Exception:
        return ""

# ----------------------------
# Local inference: title / authors from first N pages
# ----------------------------
_STOPLINE_RE = re.compile(
    r"(abstract|keywords|jel|introduction|1\.?\s+introduction|contents|table of contents|references)\b",
    flags=re.IGNORECASE
)

def _clean_line(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    return s

def _looks_like_name_token(tok: str) -> bool:
    # very lightweight: letters + optional dots/hyphens; avoid emails
    if "@" in tok:
        return False
    if len(tok) < 2:
        return False
    return bool(re.match(r"^[A-Za-z][A-Za-z\.\-']+$", tok))

def _extract_title_from_text(first_pages_text: str) -> Optional[str]:
    """
    Heuristic:
      - scan lines from top
      - collect candidate title block until we hit a stopline (Abstract/Keywords/etc.)
      - pick the "best" line: longest reasonable line with many letters
    """
    text = (first_pages_text or "").strip()
    if not text:
        return None

    lines = [_clean_line(l) for l in text.splitlines()]
    lines = [l for l in lines if l]

    # Only look early
    lines = lines[:200]

    # build candidates until Abstract-ish marker
    candidates = []
    for l in lines:
        if _STOPLINE_RE.search(l):
            break
        # skip obvious junk
        ll = l.lower()
        if any(bad in ll for bad in _BAD_TITLE_SUBSTRINGS):
            continue
        if re.search(r"\bdoi\b|arxiv", ll):
            continue
        if len(l) < 8:
            continue
        if len(l) > 220:
            continue
        # avoid lines that are mostly numbers/symbols
        alpha_ratio = sum(ch.isalpha() for ch in l) / max(len(l), 1)
        if alpha_ratio < 0.55:
            continue
        candidates.append(l)

        # if we already have a few good candidates, we can keep going a bit but not too much
        if len(candidates) >= 15:
            break

    if not candidates:
        return None

    # choose best by score (letters-heavy and reasonably long)
    def score(line: str) -> float:
        alpha = sum(ch.isalpha() for ch in line)
        return alpha + min(len(line), 120) * 0.2

    best = sorted(candidates, key=score, reverse=True)[0].strip()
    best = best[:200]
    return best or None

def _extract_authors_from_text(first_pages_text: str) -> List[str]:
    """
    Heuristic:
      - find a likely authors line near the title block (early lines)
      - parse names separated by commas / 'and' / '&'
      - keep 1..10 authors, drop affiliations/emails
    """
    text = (first_pages_text or "").strip()
    if not text:
        return []

    lines = [_clean_line(l) for l in text.splitlines()]
    lines = [l for l in lines if l]
    lines = lines[:200]

    # find a likely "authors line": after a plausible title, before Abstract
    # We'll just scan early: pick first line that has multiple name-like tokens and separators.
    author_line = None
    for l in lines[:60]:
        ll = l.lower()
        if _STOPLINE_RE.search(l):
            break
        if "@" in l:
            # might be author line with emails; still usable but we'll strip emails later
            pass

        # must contain separators or multiple capitalized tokens
        if ("," in l) or (" and " in ll) or (" & " in l):
            # check token quality
            toks = re.split(r"[\s,;&]+", l)
            nameish = sum(1 for t in toks if _looks_like_name_token(t))
            if nameish >= 3:
                author_line = l
                break

    if not author_line:
        return []

    # remove emails
    author_line = re.sub(r"\S+@\S+", "", author_line)
    author_line = re.sub(r"\s+", " ", author_line).strip()

    # split into people-ish segments
    parts = re.split(r"\s*(?:,|;|\band\b|&)\s*", author_line, flags=re.IGNORECASE)
    parts = [_clean_line(p) for p in parts if _clean_line(p)]

    authors: List[str] = []
    for p in parts:
        # drop affiliation-like chunks
        if len(p) > 60:
            continue
        if re.search(r"\b(university|department|school|institute|center|centre|lab)\b", p, flags=re.IGNORECASE):
            continue
        # require at least 2 name-like tokens
        toks = p.split()
        if sum(1 for t in toks if _looks_like_name_token(t)) >= 2:
            authors.append(p)

    # de-dup while preserving order
    seen = set()
    out = []
    for a in authors:
        key = a.lower()
        if key in seen:
            continue
        seen.add(key)
        out.append(a)

    return out[:10]

# ----------------------------
# OpenAI JSON helper (Responses API preferred)
# ----------------------------
def _openai_text_json(openai_client, model: str, system: str, user: str, temperature: float = 0.2) -> str:
    """
    Returns raw text response (expected JSON). Robust across SDK variants.
    """
    if hasattr(openai_client, "responses"):
        r = openai_client.responses.create(
            model=model,
            temperature=temperature,
            input=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        )

        out = (getattr(r, "output_text", None) or "").strip()
        if out:
            return out

        try:
            chunks = []
            for item in getattr(r, "output", []) or []:
                for c in getattr(item, "content", []) or []:
                    if isinstance(c, dict):
                        if c.get("type") == "output_text" and "text" in c:
                            chunks.append(c["text"])
                    else:
                        if getattr(c, "type", None) == "output_text" and getattr(c, "text", None):
                            chunks.append(c.text)
            return "\n".join(chunks).strip()
        except Exception:
            return str(r)

    if hasattr(openai_client, "chat") and hasattr(openai_client.chat, "completions"):
        r = openai_client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        )
        return (r.choices[0].message.content or "").strip()

    raise AttributeError("openai_client does not support responses or chat.completions.")

def parse_json_object_loose(text: str) -> dict:
    if not text or not text.strip():
        raise ValueError("Empty response text (cannot parse JSON).")

    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON object found in response text.")
    return json.loads(m.group(0))

def _repair_metadata_with_openai(
    pdf_path: "Path",
    filename: str,
    extracted_text: str,
    current: Dict[str, Any],
) -> Dict[str, Any]:
    if not LLM_REPAIR_ENABLED:
        return {}

    openai_client = globals().get("openai_client")
    if openai_client is None:
        raise NameError("openai_client is not defined")

    txt = (extracted_text or "").strip()
    if len(txt) > LLM_MAX_CHARS:
        txt = txt[:LLM_MAX_CHARS]

    system = "You are a precise academic metadata extractor. Output JSON only."
    user = f"""
Return JSON ONLY with keys:
- title: string (clean paper title)
- authors: array of strings (best-effort; empty allowed)
- year: integer or null (publication year if confidently present)

Rules:
- Infer from the most likely title/authors block near the beginning.
- Avoid acknowledgements, author contribution notes, 'Microsoft Word - ...', DOI strings as titles.
- Authors should be personal names (not affiliations).

Context:
- Filename: {filename}
- Current extracted title: {current.get('title') or ""}
- Current extracted authors: {current.get('authors') or []}
- Current DOI: {current.get('doi') or ""}
- Current arXiv: {current.get('arxiv_id') or ""}

First pages text:
{txt}
""".strip()

    raw = _openai_text_json(
        openai_client=openai_client,
        model=OPENAI_MODEL,
        system=system,
        user=user,
        temperature=OPENAI_TEMPERATURE,
    )
    obj = parse_json_object_loose(raw)

    patch: Dict[str, Any] = {}
    title = (obj.get("title") or "").strip()
    if title:
        patch["title"] = title[:200]

    authors = obj.get("authors")
    if isinstance(authors, list):
        cleaned = []
        for a in authors:
            s = str(a).strip()
            if s:
                cleaned.append(s)
        patch["authors"] = cleaned

    year = obj.get("year", None)
    if isinstance(year, int) and 1800 <= year <= (datetime.now().year + 1):
        patch["year"] = year
    else:
        patch["year"] = None

    return patch

# ----------------------------
# Main extractor
# ----------------------------
def _ocr_first_pages_text(pdf_path: Path, pages: int = 2, lang: str = "eng") -> str:
    """
    OCR fallback for image-based PDFs.
    Requires: pdf2image, pytesseract (+ poppler, tesseract installed on OS).
    Returns empty string if OCR isn't available.
    """
    try:
        import pytesseract
        from pdf2image import convert_from_path
    except Exception as e:
        logger.warning(f"[{run_id}] OCR libs not available; skipping OCR. err={e}")
        return ""

    try:
        images = convert_from_path(
            str(pdf_path),
            first_page=1,
            last_page=min(pages, 6),
        )
        chunks = []
        for img in images:
            txt = pytesseract.image_to_string(img, lang=lang) or ""
            if txt.strip():
                chunks.append(txt)
        return "\n\n".join(chunks).strip()
    except Exception as e:
        logger.warning(f"[{run_id}] OCR failed; skipping OCR. err={e}")
        return ""


def extract_pdf_metadata(pdf_path: "Path") -> Dict[str, Any]:
    result: Dict[str, Any] = {
        "doi": None,
        "arxiv_id": None,
        "title": None,
        "authors": [],
        "year": None,
        "authors_year": "",
        "raw_text_sample": "",
        "extraction_errors": [],
        "llm_used": False,
        "llm_error": None,
    }

    # ---------- Local extraction (PyPDF2 metadata + first page sample) ----------
    try:
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            metadata = reader.metadata

            if metadata:
                if metadata.get("/Title"):
                    result["title"] = str(metadata["/Title"]).strip()

                if metadata.get("/Author"):
                    author_str = str(metadata["/Author"]).strip()
                    authors = re.split(r"[;,]|\band\b", author_str, flags=re.IGNORECASE)
                    result["authors"] = [a.strip() for a in authors if a.strip()]

                if metadata.get("/Subject"):
                    subject = str(metadata["/Subject"])
                    doi_match = re.search(r"10\.\d{4,}/[-._;()/:A-Z0-9]+", subject, re.IGNORECASE)
                    if doi_match:
                        result["doi"] = doi_match.group(0)

            # page1 sample for debugging + DOI/arXiv quick wins
            if len(reader.pages) > 0:
                try:
                    text1 = reader.pages[0].extract_text() or ""
                    if text1:
                        result["raw_text_sample"] = text1[:500]

                        if not result["doi"]:
                            doi_match = re.search(r"10\.\d{4,}/[-._;()/:A-Z0-9]+", text1, re.IGNORECASE)
                            if doi_match:
                                result["doi"] = doi_match.group(0)

                        arxiv_match = re.search(r"arXiv:(\d{4}\.\d{4,5})(v\d+)?", text1, re.IGNORECASE)
                        if arxiv_match:
                            result["arxiv_id"] = arxiv_match.group(1)

                except Exception as e:
                    msg = f"Text extraction failed (page1): {e}"
                    result["extraction_errors"].append(msg)
                    logger.warning(f"[{run_id}] {msg}")

    except Exception as e:
        msg = f"PDF read failed: {e}"
        result["extraction_errors"].append(msg)
        logger.error(f"[{run_id}] {msg} for {pdf_path.name}")

    # ---------- Stronger local inference from first 2 pages ----------
    try:
        first_pages_text = _extract_first_pages_text(pdf_path, pages=LOCAL_PAGES_FOR_INFERENCE)

        # Improve title if missing/suspicious
        if _is_suspicious_title(result.get("title")):
            t2 = _extract_title_from_text(first_pages_text)
            if t2:
                result["title"] = t2

        # Improve authors if empty or obviously bad
        if not isinstance(result.get("authors"), list) or len(result["authors"]) == 0:
            a2 = _extract_authors_from_text(first_pages_text)
            if a2:
                result["authors"] = a2

    except Exception as e:
        logger.warning(f"[{run_id}] Local 2-page inference failed; continuing. err={e}")

    # ---------- Filename fallback for arXiv ----------
    if not result["arxiv_id"]:
        filename = pdf_path.stem
        arxiv_match = re.search(r"(\d{4}\.\d{4,5})", filename)
        if arxiv_match:
            result["arxiv_id"] = arxiv_match.group(1)

    # ---------- Filename fallback for title ----------
    if not result["title"]:
        filename = pdf_path.stem
        cleaned = re.sub(r"\d{4}\.\d{4,5}(v\d+)?", "", filename)
        cleaned = re.sub(r"[_-]+", " ", cleaned)
        cleaned = re.sub(r"\s+", " ", cleaned).strip()
        result["title"] = (cleaned[:200] if cleaned else filename[:200])

    # ---------- Conditional OpenAI repair ----------
    try:
        def _likely_bad_authors(authors: list) -> bool:
            if not authors:
                return True
            if len(authors) == 1:
                # PDFメタや抽出で "Scott Cunningham" しか取れない等はよくある誤りなので修正対象に
                return True
            return False
        
        def _likely_bad_title(title: str) -> bool:
            t = (title or "").strip()
            if _is_suspicious_title(t):
                return True
            # 全小文字に近いタイトルは「本文の見出し拾い」率が高い
            letters = [c for c in t if c.isalpha()]
            if letters:
                lower_ratio = sum(1 for c in letters if c.islower()) / len(letters)
                if lower_ratio > 0.85:
                    return True
            return False
        
        suspicious = _likely_bad_title(result["title"]) or _likely_bad_authors(result["authors"])

        if LLM_REPAIR_ENABLED and suspicious:
            pages_text = _extract_first_pages_text(pdf_path, pages=LLM_PAGES_FOR_TEXT)
            
            # OCR fallback if text extraction returns empty/too short
            if not pages_text or len(pages_text.strip()) < 300:
                ocr_text = _ocr_first_pages_text(pdf_path, pages=min(2, LLM_PAGES_FOR_TEXT), lang="eng")
                if ocr_text and len(ocr_text.strip()) >= 200:
                    logger.info(f"[{run_id}] Using OCR text for OpenAI metadata repair.")
                    pages_text = ocr_text
            
            if pages_text.strip():
                patch = _repair_metadata_with_openai(
                    pdf_path=pdf_path,
                    filename=pdf_path.name,
                    extracted_text=pages_text,
                    current=result,
                )
                if patch.get("title") and not _is_suspicious_title(patch["title"]):
                    result["title"] = patch["title"]
                if isinstance(patch.get("authors"), list) and patch["authors"]:
                    result["authors"] = patch["authors"]
                result["year"] = patch.get("year", None)
                result["llm_used"] = True
            else:
                logger.warning(f"[{run_id}] OpenAI repair skipped (no extractable text, OCR also unavailable/empty).")


    
    except Exception as e:
        result["llm_error"] = str(e)
        logger.warning(f"[{run_id}] OpenAI repair failed; continuing with local metadata. err={e}")

    # ---------- authors_year convenience ----------
    if result.get("authors"):
        first = result["authors"][0]
        if result.get("year"):
            result["authors_year"] = f"{first} et al., {result['year']}"
        else:
            result["authors_year"] = f"{first}"
    else:
        result["authors_year"] = ""

    # ---------- Log ----------
    logger.info(f"[{run_id}] Metadata extraction complete for {pdf_path.name}:")
    logger.info(f"  DOI: {result['doi'] or '(none)'}")
    logger.info(f"  arXiv: {result['arxiv_id'] or '(none)'}")
    logger.info(f"  Title: {(result['title'] or '')[:50]}...")
    logger.info(f"  Authors: {len(result['authors'])} found")
    if result.get("year"):
        logger.info(f"  Year: {result['year']}")
    if result.get("llm_used"):
        logger.info(f"  LLM repair: used (model={OPENAI_MODEL})")
    if result.get("llm_error"):
        logger.warning(f"  LLM repair error: {result['llm_error']}")
    if result["extraction_errors"]:
        logger.warning(f"  Errors: {'; '.join(result['extraction_errors'])}")

    return result


logger.info(f"[{run_id}] Cell 04: extract_pdf_metadata() ready (2-page local inference + OpenAI JSON fallback)")


2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Cell 04: extract_pdf_metadata() ready (2-page local inference + OpenAI JSON fallback)


In [6]:
# ============================================================
# Cell 05 — Notion wrapper capability detection (NO hard fail) [Revised]
# ============================================================
# Overview:
#   Detect which 029 wrappers are available and callable.
#   This cell must NOT assume specific wrapper names exist.
#
# Inputs / Outputs:
#   Inputs: globals() exported by 029 (optional)
#   Outputs: notion_caps dict (in memory + update_state if available)
#
# Notes:
#   - NO schema hardcoding here (029 owns mapping)
#   - NO raising just because some wrappers are missing
#   - We only fail later when we truly must write to Notion and no create wrapper exists

import inspect
from typing import Callable, Dict, Any, Optional

def _sig(func: Optional[Callable]) -> Dict[str, Any]:
    """
    Safe signature introspection for a wrapper function.
    Returns whether callable + list of parameter names (if inspectable).
    """
    if not callable(func):
        return {"callable": False, "parameters": []}
    try:
        s = inspect.signature(func)
        return {"callable": True, "parameters": list(s.parameters.keys())}
    except Exception as e:
        return {"callable": True, "parameters": [], "inspect_error": str(e)}

def _get(name: str):
    return globals().get(name)

# --- Candidate wrapper names (adapt to what 029 exports) ---
CREATE_CANDIDATES = [
    "create_paper_inbox",
    "create_paper",            # fallback
    "create_paper_record",
    "create_paper_page",
]

DEDUP_CANDIDATES = [
    "find_duplicate_paper",         # preferred generic (often the real one)
    "find_duplicate_by_doi",
    "find_duplicate_by_arxiv_id",   # preferred arxiv name
    "find_duplicate_by_arxiv",      # older/alt name
    "find_duplicate_by_title",
]

# --- Collect functions ---
create_funcs = {name: _get(name) for name in CREATE_CANDIDATES}
dedup_funcs  = {name: _get(name) for name in DEDUP_CANDIDATES}

# --- Build capability map ---
notion_caps: Dict[str, Any] = {
    "create": {name: _sig(fn) for name, fn in create_funcs.items()},
    "dedup":  {name: _sig(fn) for name, fn in dedup_funcs.items()},
}

# --- Pick preferred create wrapper ---
preferred_create = None
for name in ["create_paper_inbox", "create_paper", "create_paper_record", "create_paper_page"]:
    if notion_caps["create"].get(name, {}).get("callable"):
        preferred_create = name
        break

# --- Pick preferred dedup wrappers ---
preferred_generic_dedup = "find_duplicate_paper" if notion_caps["dedup"].get("find_duplicate_paper", {}).get("callable") else None

preferred_arxiv_dedup = None
for name in ["find_duplicate_by_arxiv_id", "find_duplicate_by_arxiv"]:
    if notion_caps["dedup"].get(name, {}).get("callable"):
        preferred_arxiv_dedup = name
        break

preferred_doi_dedup = "find_duplicate_by_doi" if notion_caps["dedup"].get("find_duplicate_by_doi", {}).get("callable") else None
preferred_title_dedup = "find_duplicate_by_title" if notion_caps["dedup"].get("find_duplicate_by_title", {}).get("callable") else None

notion_caps["preferred"] = {
    "create": preferred_create,
    "dedup_generic": preferred_generic_dedup,
    "dedup_doi": preferred_doi_dedup,
    "dedup_arxiv": preferred_arxiv_dedup,
    "dedup_title": preferred_title_dedup,
}

# --- Log summary ---
logger.info(f"[{run_id}] Notion wrapper detection:")
logger.info(f"[{run_id}]  preferred create:      {notion_caps['preferred']['create']}")
logger.info(f"[{run_id}]  preferred dedup:       {notion_caps['preferred']['dedup_generic']}")
logger.info(f"[{run_id}]  preferred dedup doi:   {notion_caps['preferred']['dedup_doi']}")
logger.info(f"[{run_id}]  preferred dedup arxiv: {notion_caps['preferred']['dedup_arxiv']}")
logger.info(f"[{run_id}]  preferred dedup title: {notion_caps['preferred']['dedup_title']}")

# --- (Optional) log signatures for quick debugging ---
try:
    if notion_caps["preferred"]["create"]:
        logger.info(f"[{run_id}]  create signature params: {notion_caps['create'][notion_caps['preferred']['create']]['parameters']}")
    if notion_caps["preferred"]["dedup_generic"]:
        logger.info(f"[{run_id}]  dedup_generic params: {notion_caps['dedup'][notion_caps['preferred']['dedup_generic']]['parameters']}")
except Exception:
    pass

# --- Persist capabilities (optional) ---
try:
    update_state("notion_caps", notion_caps)
except Exception:
    pass


2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Notion wrapper detection:
2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63]  preferred create:      create_paper_inbox
2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63]  preferred dedup:       find_duplicate_paper
2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63]  preferred dedup doi:   None
2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63]  preferred dedup arxiv: None
2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63]  preferred dedup title: None
2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63]  create signature para

In [7]:
# ============================================================
# Cell 06 — Deduplication checker (capability-aware + safe) [Revised]
# ============================================================
# Overview:
#   Capability-aware dedup checks. Only calls Notion dedup wrappers if callable.
#   Prefers generic dedup (find_duplicate_paper) when available, then falls back
#   to DOI → arXiv → title wrappers.
#
# Inputs / Outputs:
#   Inputs: metadata dict, notion_caps (optional), normalize_title (from Cell 03)
#   Outputs: dict {is_duplicate, match_type, page_id, title, error, checked}
#
# Notes:
#   - Never calls None / missing functions
#   - Supports 029 "find_duplicate_paper" (preferred in your Notion layer)
#   - If no dedup wrappers exist, it returns no-duplicate with checked=[]

import re
from typing import Dict, Any, Optional, Callable

def _get_callable(name: Optional[str]) -> Optional[Callable]:
    fn = globals().get(name) if name else None
    return fn if callable(fn) else None

def _pick_dedup_fns():
    caps = globals().get("notion_caps")
    preferred = (caps or {}).get("preferred", {}) if isinstance(caps, dict) else {}

    generic_name = preferred.get("dedup_generic") or ("find_duplicate_paper" if callable(globals().get("find_duplicate_paper")) else None)
    doi_name     = preferred.get("dedup_doi") or ("find_duplicate_by_doi" if callable(globals().get("find_duplicate_by_doi")) else None)
    arxiv_name   = preferred.get("dedup_arxiv") or ("find_duplicate_by_arxiv_id" if callable(globals().get("find_duplicate_by_arxiv_id")) else None)
    title_name   = preferred.get("dedup_title") or ("find_duplicate_by_title" if callable(globals().get("find_duplicate_by_title")) else None)

    return {
        "generic": _get_callable(generic_name),
        "doi": _get_callable(doi_name),
        "arxiv": _get_callable(arxiv_name),
        "title": _get_callable(title_name),
        "names": {"generic": generic_name, "doi": doi_name, "arxiv": arxiv_name, "title": title_name},
    }

def _extract_match_fields(match: Any) -> Dict[str, Any]:
    if match is None:
        return {"page_id": None, "title": None}
    if isinstance(match, list) and match:
        match = match[0]
    if isinstance(match, dict):
        page_id = match.get("id") or match.get("page_id")
        title = match.get("title") or match.get("name")
        props = match.get("properties")
        if title is None and isinstance(props, dict):
            for k in ["Title", "title", "Name", "Paper", "Paper Title"]:
                v = props.get(k)
                if isinstance(v, dict):
                    tarr = v.get("title") or v.get("rich_text")
                    if isinstance(tarr, list) and tarr:
                        title = tarr[0].get("plain_text")
                        break
        return {"page_id": page_id, "title": title}
    return {"page_id": None, "title": None}

def _build_source_uid(metadata: Dict[str, Any], pdf_path: Optional[str] = None) -> Optional[str]:
    doi = (metadata.get("doi") or "").strip()
    arxiv = (metadata.get("arxiv_id") or "").strip()
    if doi:
        return f"doi:{doi.lower()}"
    if arxiv:
        return f"arxiv:{arxiv}"
    if pdf_path:
        return f"file:{pdf_path}"
    return None

def check_for_duplicate(metadata: Dict[str, Any]) -> Dict[str, Any]:
    fns = _pick_dedup_fns()

    result = {
        "is_duplicate": False,
        "match_type": None,
        "page_id": None,
        "title": None,
        "error": None,
        "checked": [],
    }

    title_raw = (metadata.get("title") or "").strip()
    doi = (metadata.get("doi") or "").strip()
    arxiv_id = (metadata.get("arxiv_id") or "").strip()

    # Normalize title if possible
    norm_fn = globals().get("normalize_title")
    title_norm = norm_fn(title_raw) if callable(norm_fn) else re.sub(r"\s+", " ", title_raw).strip().lower()

    # --- 0) Preferred generic dedup (if available) ---
    # Signature in your 029 example:
    # find_duplicate_paper(name, dedup_key, source_uid, pdf_link=None)
    if fns["generic"] and title_raw:
        try:
            source_uid = _build_source_uid(metadata)
            # Prefer dedup_key based on source_uid when available; else title norm
            dedup_seed = source_uid or title_norm or title_raw
            dedup_key = make_dedup_key(dedup_seed) if callable(globals().get("make_dedup_key")) else None

            result["checked"].append(f"generic:{fns['names']['generic']}")

            match = fns["generic"](
                name=title_raw,
                dedup_key=dedup_key,
                source_uid=source_uid,
                pdf_link=None,
            )
            # Many implementations return tuple: (is_dup, existing_id, reason)
            if isinstance(match, tuple) and len(match) >= 2:
                is_dup = bool(match[0])
                existing_id = match[1]
                reason = match[2] if len(match) >= 3 else None
                if is_dup and existing_id:
                    result.update({"is_duplicate": True, "match_type": f"generic:{reason or 'dedup_key'}", "page_id": existing_id, "title": None})
                    return result
            elif match:
                fields = _extract_match_fields(match)
                result.update({"is_duplicate": True, "match_type": "generic", **fields})
                return result

        except Exception as e:
            msg = f"generic dedup failed via {fns['names']['generic']}: {e}"
            logger.warning(f"[{run_id}] {msg}")
            result["error"] = (result["error"] + "; " if result["error"] else "") + msg

    # --- 1) DOI ---
    if doi and fns["doi"]:
        result["checked"].append(f"doi:{fns['names']['doi']}")
        try:
            match = fns["doi"](doi)
            if match:
                fields = _extract_match_fields(match)
                result.update({"is_duplicate": True, "match_type": "doi", **fields})
                return result
        except Exception as e:
            msg = f"DOI dedup failed via {fns['names']['doi']}: {e}"
            logger.warning(f"[{run_id}] {msg}")
            result["error"] = (result["error"] + "; " if result["error"] else "") + msg

    # --- 2) arXiv ---
    if arxiv_id and fns["arxiv"]:
        result["checked"].append(f"arxiv:{fns['names']['arxiv']}")
        try:
            match = fns["arxiv"](arxiv_id)
            if match:
                fields = _extract_match_fields(match)
                result.update({"is_duplicate": True, "match_type": "arxiv", **fields})
                return result
        except Exception as e:
            msg = f"arXiv dedup failed via {fns['names']['arxiv']}: {e}"
            logger.warning(f"[{run_id}] {msg}")
            result["error"] = (result["error"] + "; " if result["error"] else "") + msg

    # --- 3) Title ---
    if title_norm and fns["title"]:
        result["checked"].append(f"title:{fns['names']['title']}")
        try:
            match = fns["title"](title_norm)
            if match:
                fields = _extract_match_fields(match)
                result.update({"is_duplicate": True, "match_type": "title", **fields})
                return result
        except Exception as e:
            msg = f"title dedup failed via {fns['names']['title']}: {e}"
            logger.warning(f"[{run_id}] {msg}")
            result["error"] = (result["error"] + "; " if result["error"] else "") + msg

    logger.info(f"[{run_id}] No duplicate found. checked={result['checked']}")
    return result


logger.info(f"[{run_id}] Cell 06: Dedup checker ready (capability-aware)")


2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Cell 06: Dedup checker ready (capability-aware)


In [8]:
# ============================================================
# Cell 07 — Slide Spec (IR) generation + Gemini slide rendering (JP body, EN header)
#         (015 Step7+8 port, upgraded layout + optional OpenAI structuring)
# ============================================================
# Goal:
#   - Header: Title / Authors in English
#   - Body (Japanese): ①RQ ②既存研究 ③新規性 ④データ・手法 ⑤結果・示唆
#   - Add small icons (emoji) + clean 16:9 slide composition
#
# Runtime:
#   - Gemini is required for image generation (GEMINI_API_KEY).
#   - OpenAI is OPTIONAL for generating structured JP content.
#     If openai_client is missing or fails, we fall back to “空欄/不明” hints.

import os
import json
import time
import re
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, Optional, Tuple, List

# ----------------------------
# Gemini configuration
# ----------------------------
GEMINI_IMAGE_MODEL: str = "gemini-3-pro-image-preview"

os.environ.pop("GOOGLE_API_KEY", None)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if GEMINI_API_KEY is None:
    raise ValueError(
        "GEMINI_API_KEY could not be loaded from env.txt (via 028). "
        "This key is required for slide image generation."
    )
else:
    logger.info(f"[{run_id}] ✅ GEMINI_API_KEY loaded successfully")

from google import genai
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# ----------------------------
# Optional OpenAI configuration (for JP structured summary)
# ----------------------------
OPENAI_ENABLED = True
OPENAI_MODEL = globals().get("OPENAI_MODEL", "gpt-4.1-mini")
OPENAI_TEMPERATURE = float(globals().get("OPENAI_TEMPERATURE", 0.2))

# How much paper text to feed OpenAI (from PDF extraction)
OPENAI_MAX_CHARS = 45000
OPENAI_PAGES_FOR_TEXT = 6  # increase if your PDFs have sparse text

# ----------------------------
# Helpers
# ----------------------------
try:
    import fitz  # PyMuPDF
    _HAS_FITZ = True
except Exception:
    _HAS_FITZ = False

def safe_slug(s: str, max_len: int = 90) -> str:
    s = (s or "").strip()
    s = re.sub(r"[^\w\-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:max_len] if s else "paper"

def _ensure_dirs():
    slides_path.mkdir(parents=True, exist_ok=True)
    spec_dir = slides_path / "_spec"
    spec_dir.mkdir(parents=True, exist_ok=True)
    return spec_dir

def _now_tag() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def _best_source_uid(metadata: Dict[str, Any], pdf_path: Path) -> str:
    doi = (metadata.get("doi") or "").strip()
    arxiv = (metadata.get("arxiv_id") or "").strip()
    if doi:
        return f"doi:{doi.lower()}"
    if arxiv:
        return f"arxiv:{arxiv}"
    return f"file:{pdf_path.stem}"

def _wrap(text: str, max_chars: int = 70) -> str:
    text = (text or "").strip()
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 1].rstrip() + "…"

def _englishish_title(title: str) -> str:
    """
    Trust Cell 04 repaired title.
    Only normalize whitespace, do NOT truncate aggressively.
    """
    t = re.sub(r"\s+", " ", (title or "").strip())
    return t if t else "Untitled Paper"


def _englishish_authors(metadata: Dict[str, Any]) -> str:
    """
    Prefer authors_year if available (more informative for header).
    Fallback to authors list.
    """
    ay = (metadata.get("authors_year") or "").strip()
    if ay:
        return ay

    authors = metadata.get("authors") or []
    if isinstance(authors, list) and authors:
        if len(authors) == 1:
            return str(authors[0]).strip()
        if len(authors) == 2:
            return f"{authors[0]} & {authors[1]}"
        return f"{authors[0]} et al."

    return "Unknown authors"


def _extract_pages_text(pdf_path: Path, pages: int = 6) -> str:
    # 1) Try normal text extraction first (fast)
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path)
            chunks = []
            for i in range(min(pages, len(doc))):
                txt = doc[i].get_text("text") or ""
                if txt.strip():
                    chunks.append(txt)
            doc.close()
            joined = "\n\n".join(chunks).strip()

            # Heuristic: if too short, likely image-based -> OCR fallback
            if len(joined) >= 300:
                return joined
            logger.warning(f"[{run_id}] Text extracted but too short ({len(joined)} chars). Trying OCR fallback...")
        except Exception as e:
            logger.warning(f"[{run_id}] fitz text extraction failed; trying OCR fallback. err={e}")

    # 2) OCR fallback (slow)
    try:
        import pytesseract
        from pdf2image import convert_from_path

        images = convert_from_path(str(pdf_path), first_page=1, last_page=min(pages, 6))
        ocr_chunks = []
        for img in images:
            ocr_txt = pytesseract.image_to_string(img, lang="eng")  # 必要なら "jpn+eng"
            if ocr_txt.strip():
                ocr_chunks.append(ocr_txt)

        ocr_joined = "\n\n".join(ocr_chunks).strip()
        if ocr_joined:
            logger.info(f"[{run_id}] OCR extracted text ({len(ocr_joined)} chars).")
            return ocr_joined
    except Exception as e:
        logger.warning(f"[{run_id}] OCR fallback failed (optional). err={e}")

    # 3) last fallback
    return (metadata.get("raw_text_sample") or "").strip()

def _openai_text(openai_client, model: str, system: str, user: str, temperature: float = 0.2) -> str:
    """
    Robust helper (Responses API preferred). Returns raw text.
    """
    if hasattr(openai_client, "responses"):
        r = openai_client.responses.create(
            model=model,
            temperature=temperature,
            input=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        )
        out = (getattr(r, "output_text", None) or "").strip()
        if out:
            return out
        try:
            chunks = []
            for item in getattr(r, "output", []) or []:
                for c in getattr(item, "content", []) or []:
                    if isinstance(c, dict):
                        if c.get("type") == "output_text" and "text" in c:
                            chunks.append(c["text"])
                    else:
                        if getattr(c, "type", None) == "output_text" and getattr(c, "text", None):
                            chunks.append(c.text)
            return "\n".join(chunks).strip()
        except Exception:
            return str(r)

    if hasattr(openai_client, "chat") and hasattr(openai_client.chat, "completions"):
        r = openai_client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        )
        return (r.choices[0].message.content or "").strip()

    raise AttributeError("openai_client does not support responses or chat.completions.")

def _parse_json_loose(text: str) -> dict:
    if not text or not text.strip():
        raise ValueError("Empty response text.")
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not m:
            raise ValueError("No JSON object found.")
        return json.loads(m.group(0))

def _generate_jp_structured_summary(pdf_path: Path, metadata: Dict[str, Any]) -> Dict[str, str]:
    """
    Use OpenAI (optional) to produce JP body fields.
    If uncertain, model should output '不明' rather than hallucinating.
    """
    if not OPENAI_ENABLED:
        return {}

    openai_client = globals().get("openai_client")
    if openai_client is None:
        raise NameError("openai_client is not defined")

    txt = _extract_pages_text(pdf_path, metadata, pages=OPENAI_PAGES_FOR_TEXT)
    txt = (txt or "").strip()
    if not txt:
        return {}

    if len(txt) > OPENAI_MAX_CHARS:
        txt = txt[:OPENAI_MAX_CHARS]

    system = "You are a careful research assistant. Output JSON only. If unknown, write '不明'."
    user = f"""
以下の論文テキスト（冒頭ページ中心）から、スライド1枚用の要約を作ってください。
**必ずJSONのみ**で返し、推測で断定しないでください。不明な場合は「不明」。

出力キー（すべて日本語）:
- rq: 研究クエスチョン（1文）
- prior: 既存研究は何か（1-2文）
- novelty: 新規性・貢献（1-2文）
- data_method: データと分析手法（1-2文）
- results: 結果・何が言えたか（1-2文）
- takeaway: 一番の示唆（超短い1文）

参考メタ情報:
- title: {metadata.get("title") or ""}
- doi: {metadata.get("doi") or ""}
- arxiv_id: {metadata.get("arxiv_id") or ""}

論文テキスト:
{txt}
""".strip()

    raw = _openai_text(openai_client, OPENAI_MODEL, system, user, temperature=OPENAI_TEMPERATURE)
    obj = _parse_json_loose(raw)

    def _s(k: str) -> str:
        v = obj.get(k)
        v = "" if v is None else str(v).strip()
        return v

    return {
        "rq": _s("rq"),
        "prior": _s("prior"),
        "novelty": _s("novelty"),
        "data_method": _s("data_method"),
        "results": _s("results"),
        "takeaway": _s("takeaway"),
    }

# ----------------------------
# IR Builder (JP body + EN header)
# ----------------------------
def build_slide_spec_ir(pdf_path: Path, metadata: Dict[str, Any]) -> Dict[str, Any]:
    title_en = _englishish_title(metadata.get("title") or pdf_path.stem)
    authors_en = _englishish_authors(metadata)

    # JP body fields (OpenAI optional)
    jp = {}
    jp_error = None
    try:
        jp = _generate_jp_structured_summary(pdf_path, metadata) if OPENAI_ENABLED else {}
    except Exception as e:
        jp_error = str(e)
        logger.warning(f"[{run_id}] JP summary via OpenAI failed; continuing with placeholders. err={e}")

    # placeholders if empty
    def _jp_or(key: str, placeholder: str) -> str:
        v = (jp.get(key) or "").strip()
        return v if v else placeholder

    ir = {
        "version": "ir_v2_jp_body_en_header",
        "source": {
            "pdf_name": pdf_path.name,
            "source_uid": _best_source_uid(metadata, pdf_path),
            "doi": metadata.get("doi"),
            "arxiv_id": metadata.get("arxiv_id"),
        },
        "slide": {
            # Header must be English
            "title_en": _wrap(title_en, 140),
            "authors_en": _wrap(authors_en, 80),

            # Body must be Japanese (5 blocks)
            "blocks_jp": [
                {
                    "icon": "🎯",
                    "heading_jp": "研究クエスチョン",
                    "text_jp": _jp_or("rq", "不明（テキスト抽出が不足している可能性）"),
                },
                {
                    "icon": "📚",
                    "heading_jp": "既存研究",
                    "text_jp": _jp_or("prior", "不明"),
                },
                {
                    "icon": "✨",
                    "heading_jp": "新規性・貢献",
                    "text_jp": _jp_or("novelty", "不明"),
                },
                {
                    "icon": "🧪",
                    "heading_jp": "データ・分析手法",
                    "text_jp": _jp_or("data_method", "不明"),
                },
                {
                    "icon": "📈",
                    "heading_jp": "結果・示唆",
                    "text_jp": _jp_or("results", "不明"),
                },
            ],
            "takeaway_jp": _jp_or("takeaway", "不明"),

            "badges": [b for b in [
                ("DOI" if metadata.get("doi") else None),
                ("arXiv" if metadata.get("arxiv_id") else None),
            ] if b],
            "style": {
                "theme": "clean_academic",
                "layout": "header + 2col cards + takeaway bar",
                "density": "high_but_readable",
                "color_policy": "neutral",
                "aspect": "16:9",
                "language_policy": "Header EN / Body JP",
            },
            "runtime_notes": {
                "openai_used_for_jp": bool(jp),
                "openai_error": jp_error,
                "openai_model": OPENAI_MODEL if bool(jp) else None,
            }
        }
    }
    return ir

# ----------------------------
# Gemini renderer (single slide image)
# ----------------------------

def render_slide_with_gemini(ir: Dict[str, Any]) -> Tuple[bytes, str]:
    ir_json = json.dumps(ir, ensure_ascii=False, indent=2)
    title_fixed = ir["slide"]["title_en"]
    authors_fixed = ir["slide"]["authors_en"]
    prompt = f"""
You are a professional academic slide designer.

Create exactly ONE 16:9 slide image based on the Slide Spec (IR) below.

CRITICAL FIXED TEXT (MUST COPY VERBATIM, DO NOT REPHRASE):
- Title (English): {title_fixed}
- Authors (English): {authors_fixed}

Hard requirements:
- Header (top): MUST show the exact Title and Authors above (verbatim).
  - Do NOT shorten, rewrite, translate, or infer from paper text.
  - If the strings are long, reduce font size and wrap to 2 lines, but keep wording identical.
- Body: Japanese only.
- Use a clean academic theme. White background, subtle dividers, plenty of padding.
- Layout suggestion:
  - Top header bar (title + authors)
  - Body area: 2-column grid of 5 small "cards" (cards can wrap; e.g., 3 on left, 2 on right)
  - Bottom: a compact "Takeaway" strip (Japanese).
- Each card MUST include:
  - small icon (use the provided emoji or draw a simple icon-like glyph)
  - Japanese heading
  - Japanese text (2–3 lines max; truncate with ellipsis if too long)
- No long paragraphs. No bullet symbols like "•" or "-".
- Do NOT invent details. If text says "不明", keep it as-is (do not replace).
- Do not add citations, URLs, journal names, or years unless present in the IR.
- Output should be presentation-ready (sharp text).

Slide Spec (IR):
{ir_json}
""".strip()

    resp = gemini_client.models.generate_content(
        model=GEMINI_IMAGE_MODEL,
        contents=prompt
    )

    image_bytes = None

    # candidates -> content.parts -> inline_data.data
    try:
        for cand in getattr(resp, "candidates", []) or []:
            content = getattr(cand, "content", None)
            if not content:
                continue
            for part in getattr(content, "parts", []) or []:
                inline = getattr(part, "inline_data", None)
                if inline and getattr(inline, "data", None):
                    image_bytes = inline.data
                    break
            if image_bytes is not None:
                break
    except Exception:
        pass

    # resp.parts -> inline_data.data
    if image_bytes is None:
        try:
            for part in getattr(resp, "parts", []) or []:
                inline = getattr(part, "inline_data", None)
                if inline and getattr(inline, "data", None):
                    image_bytes = inline.data
                    break
        except Exception:
            pass

    if image_bytes is None:
        raise RuntimeError("Gemini returned no inline image bytes (unexpected response shape).")

    return image_bytes, prompt

# ----------------------------
# Public entry: PDF -> IR -> Gemini slide -> save artifacts
# ----------------------------
def generate_slide_artifact(pdf_path: Path, metadata: Dict[str, Any]) -> Optional[Path]:
    spec_dir = _ensure_dirs()

    title_for_name = metadata.get("title") or pdf_path.stem
    dedup_key = make_dedup_key(_best_source_uid(metadata, pdf_path)) or make_dedup_key(title_for_name) or safe_slug(pdf_path.stem, 32)
    base = f"{safe_slug(title_for_name, 60)}__{dedup_key}__{_now_tag()}"

    slide_path = slides_path / f"{base}.png"
    ir_path = spec_dir / f"{base}__ir.json"
    prompt_path = spec_dir / f"{base}__prompt.txt"

    try:
        # Step 7: IR
        ir = build_slide_spec_ir(pdf_path, metadata)
        ir_path.write_text(json.dumps(ir, ensure_ascii=False, indent=2), encoding="utf-8")

        # Step 8: Gemini render (retry)
        last_err = None
        for attempt in range(1, 4):
            try:
                img_bytes, prompt = render_slide_with_gemini(ir)
                slide_path.write_bytes(img_bytes)
                prompt_path.write_text(prompt, encoding="utf-8")
                logger.info(f"[{run_id}] Slide artifact saved (Gemini): {slide_path.name}")
                return slide_path
            except Exception as e:
                last_err = e
                logger.warning(f"[{run_id}] Gemini render attempt {attempt}/3 failed: {e}")
                time.sleep(1.5 * attempt)

        raise RuntimeError(f"Gemini render failed after retries: {last_err}")

    except Exception as e:
        logger.error(f"[{run_id}] Slide generation failed for {pdf_path.name}: {e}")
        return None

# ----------------------------
# Optional loop test (OFF by default)
# ----------------------------
RUN_SLIDE_LOOP_TEST = False

if RUN_SLIDE_LOOP_TEST:
    logger.info(f"[{run_id}] Loop test: generating JP-body slides for all PDFs in inbox")

    pdfs = sorted(
        {p.resolve() for p in list(inbox_path.glob("*.pdf")) + list(inbox_path.glob("*.PDF")) if p.is_file()},
        key=lambda p: p.name.lower()
    )
    logger.info(f"[{run_id}] PDFs found: {len(pdfs)}")

    generated: List[str] = []
    failed: List[str] = []

    for i, p in enumerate(pdfs, 1):
        logger.info(f"[{run_id}] [{i}/{len(pdfs)}] {p.name}")
        meta = extract_pdf_metadata(p)
        sp = generate_slide_artifact(p, meta)
        if sp and sp.exists():
            generated.append(str(sp))
        else:
            failed.append(p.name)

    logger.info(f"[{run_id}] Slide loop done: generated={len(generated)} failed={len(failed)}")

    try:
        update_state("slide_generation", {"generated": generated, "failed": failed})
    except Exception:
        pass
else:
    logger.info(f"[{run_id}] Cell 07 ready (EN header + JP body cards + takeaway). RUN_SLIDE_LOOP_TEST=False")


2026-02-01 09:31:50 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] ✅ GEMINI_API_KEY loaded successfully
2026-02-01 09:31:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Cell 07 ready (EN header + JP body cards + takeaway). RUN_SLIDE_LOOP_TEST=False


In [9]:
# ============================================================
# Cell 08 — Notion record creation adapter (create_paper_inbox-aligned + self-healing fields)
# ============================================================
# What this cell does:
#   1) Extract "rich fields" (core idea / datasets / methods / findings / notes / tags etc.)
#      from PDF text using OpenAI (JSON-only). Cache per paper as notion_fields.json.
#   2) Call 029's create_paper_inbox() with the confirmed signature.
#   3) Put rich fields + local paths into `extra` (so you don't need DB schema changes).
#
# Requires:
#   - create_paper_inbox callable (from 029)
#   - openai_client initialized (if you want self-healing fields; otherwise it will degrade gracefully)
#   - artifacts_path, run_id, logger already available (from 028/029 context)
#
# NOTE:
#   Your screenshot shows Notion properties mostly empty because the wrapper only received
#   minimal fields (name/authors_year/pdf_status/dedup/run_id). This cell fixes that by
#   generating notion_fields and passing them through `extra` AND (when possible) mapping
#   to known DB properties if 029 supports it.

import json
import re
import inspect
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, Optional, List, Tuple

# ----------------------------
# Config knobs
# ----------------------------
OPENAI_MODEL = globals().get("OPENAI_MODEL", "gpt-4.1-mini")
OPENAI_TEMPERATURE = float(globals().get("OPENAI_TEMPERATURE", 0.2))
PDF_TEXT_MAX_CHARS = 120000
PDF_TEXT_MAX_PAGES = 20

# where to store per-paper artifacts
PDF_RUNS_DIRNAME = "pdf_runs"

# ----------------------------
# PDF text extraction for LLM (best-effort, no OCR)
# ----------------------------
def _extract_pdf_text_for_llm(pdf_path: Path, max_pages: int = 20, max_chars: int = 120000) -> str:
    """
    Best-effort extraction using PyMuPDF if available else PyPDF2.
    Avoid OCR; deterministic caps for runtime.
    """
    # 1) PyMuPDF (preferred)
    try:
        import fitz  # type: ignore
        doc = fitz.open(pdf_path)
        chunks: List[str] = []
        for i in range(min(max_pages, len(doc))):
            t = (doc[i].get_text("text") or "").strip()
            if t:
                chunks.append(t)
            if sum(len(x) for x in chunks) >= max_chars:
                break
        doc.close()
        out = "\n\n".join(chunks).strip()
        return out[:max_chars]
    except Exception:
        pass

    # 2) PyPDF2 fallback
    try:
        import PyPDF2  # assumed available
        chunks = []
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages[:max_pages]:
                t = (page.extract_text() or "").strip()
                if t:
                    chunks.append(t)
                if sum(len(x) for x in chunks) >= max_chars:
                    break
        out = "\n\n".join(chunks).strip()
        return out[:max_chars]
    except Exception as e:
        logger.warning(f"[{run_id}] PDF text extraction failed for LLM: {e}")
        return ""

# ----------------------------
# OpenAI helper (Responses API preferred) + loose JSON parser
# ----------------------------
def _openai_text(openai_client, model: str, system: str, user: str, temperature: float = 0.2) -> str:
    if hasattr(openai_client, "responses"):
        r = openai_client.responses.create(
            model=model,
            temperature=temperature,
            input=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        )
        out = (getattr(r, "output_text", None) or "").strip()
        if out:
            return out
        try:
            chunks = []
            for item in getattr(r, "output", []) or []:
                for c in getattr(item, "content", []) or []:
                    if isinstance(c, dict):
                        if c.get("type") == "output_text" and "text" in c:
                            chunks.append(c["text"])
                    else:
                        if getattr(c, "type", None) == "output_text" and getattr(c, "text", None):
                            chunks.append(c.text)
            return "\n".join(chunks).strip()
        except Exception:
            return str(r)

    if hasattr(openai_client, "chat") and hasattr(openai_client.chat, "completions"):
        r = openai_client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        )
        return (r.choices[0].message.content or "").strip()

    raise AttributeError("openai_client does not support responses or chat.completions.")

def _parse_json_object_loose(text: str) -> dict:
    if not text or not text.strip():
        raise ValueError("Empty response text (cannot parse JSON).")
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not m:
            raise ValueError("No JSON object found in response text.")
        return json.loads(m.group(0))

# ----------------------------
# ID helpers
# ----------------------------
def _make_source_uid(metadata: Dict[str, Any], pdf_path: Path) -> str:
    doi = (metadata.get("doi") or "").strip()
    arxiv = (metadata.get("arxiv_id") or "").strip()
    if doi:
        return f"doi:{doi.lower()}"
    if arxiv:
        return f"arxiv:{arxiv.lower()}"
    return f"file:{pdf_path.name.lower()}"

def _make_paper_dedup_key(source_uid: str) -> str:
    base = f"paper|{source_uid}"
    try:
        return make_dedup_key(base)  # from Cell 03
    except Exception:
        import hashlib
        return hashlib.sha256(base.encode("utf-8")).hexdigest()[:16]

def _to_authors_year(metadata: Dict[str, Any]) -> str:
    authors = metadata.get("authors") or []
    authors = [str(a).strip() for a in authors if str(a).strip()]
    authors_str = ", ".join(authors)

    year = metadata.get("year")
    year_str = ""
    if isinstance(year, int):
        year_str = str(year)
    elif isinstance(year, str) and re.match(r"^\d{4}$", year.strip()):
        year_str = year.strip()

    if authors_str and year_str:
        return f"{authors_str} ({year_str})"
    return authors_str or ""

# ----------------------------
# Self-healing notion_fields generator (cached per paper)
# ----------------------------
def _get_pdf_run_dir(artifacts_path: Path, dedup_key: str) -> Path:
    d = artifacts_path / PDF_RUNS_DIRNAME / dedup_key
    d.mkdir(parents=True, exist_ok=True)
    return d

def _get_or_generate_notion_fields(
    pdf_path: Path,
    metadata: Dict[str, Any],
    pdf_run_dir: Path,
    openai_model: str,
    temperature: float,
) -> Dict[str, Any]:
    """
    Creates/loads notion_fields.json (015-like) and returns dict.
    If openai_client is missing or extraction fails, returns minimal dict.
    """
    notion_fields_path = pdf_run_dir / "notion_fields.json"

    if notion_fields_path.exists():
        try:
            nf = json.loads(notion_fields_path.read_text(encoding="utf-8"))
            if isinstance(nf, dict) and nf:
                return nf
        except Exception as e:
            logger.warning(f"[{run_id}] notion_fields.json exists but failed to load; regenerating. err={e}")

    # Minimal fallback if no OpenAI client
    openai_client = globals().get("openai_client")
    if openai_client is None:
        nf = {
            "name": (metadata.get("title") or pdf_path.stem).strip() or "Untitled Paper",
            "authors_year": _to_authors_year(metadata),
            "source": "",
            "type": "",
            "core_idea": "",
            "datasets": "",
            "methods": "",
            "findings": "",
            "notes": "openai_client is not initialized; metadata-only record.",
            "tags": [],
        }
        notion_fields_path.write_text(json.dumps(nf, ensure_ascii=False, indent=2), encoding="utf-8")
        return nf

    paper_text = _extract_pdf_text_for_llm(pdf_path, max_pages=PDF_TEXT_MAX_PAGES, max_chars=PDF_TEXT_MAX_CHARS)

    if not paper_text.strip():
        nf = {
            "name": (metadata.get("title") or pdf_path.stem).strip() or "Untitled Paper",
            "authors_year": _to_authors_year(metadata),
            "source": "",
            "type": "",
            "core_idea": "",
            "datasets": "",
            "methods": "",
            "findings": "",
            "notes": "PDF text extraction empty; metadata-only record.",
            "tags": [],
        }
        notion_fields_path.write_text(json.dumps(nf, ensure_ascii=False, indent=2), encoding="utf-8")
        return nf

    system = "You are a precise research assistant who writes concise database-ready summaries. Return JSON only."
    user = f"""
Create a Literature Database entry from the paper text below.

Output language rules:
- Name (title): English (single line)
- Authors & Year: English (format: "Last, First (Year)" if possible; else best effort)
- Source: English (journal / venue) (unknown => empty string)
- Type: English (short noun phrase, unknown => empty string)
- Tags: English (JSON array of short tags, 2–6 items, Title Case; unknown => empty array)

- Core Idea: Japanese (unknown => "不明")
- Datasets: Japanese (unknown => "不明")
- Methods: Japanese (unknown => "不明")
- Findings: Japanese (unknown => "不明")
- Notes: Japanese (short; unknown => "")

Return JSON ONLY with keys:
name, authors_year, source, type, core_idea, datasets, methods, findings, notes, tags

Metadata hints:
- title_hint: {metadata.get("title") or ""}
- doi_hint: {(metadata.get("doi") or "").strip()}
- arxiv_hint: {(metadata.get("arxiv_id") or "").strip()}

Paper text (truncated):
{paper_text[:PDF_TEXT_MAX_CHARS]}
""".strip()

    out = _openai_text(openai_client, openai_model, system, user, temperature=temperature)
    nf = _parse_json_object_loose(out)

    # Guardrails
    if not isinstance(nf, dict):
        nf = {}

    nf.setdefault("name", (metadata.get("title") or pdf_path.stem).strip() or "Untitled Paper")
    nf.setdefault("authors_year", _to_authors_year(metadata))
    nf.setdefault("source", "")
    nf.setdefault("type", "")
    nf.setdefault("core_idea", "不明")
    nf.setdefault("datasets", "不明")
    nf.setdefault("methods", "不明")
    nf.setdefault("findings", "不明")
    nf.setdefault("notes", "")
    nf.setdefault("tags", [])

    if not isinstance(nf.get("tags"), list):
        nf["tags"] = []
    nf["tags"] = [str(t).strip() for t in nf["tags"] if str(t).strip()]

    notion_fields_path.write_text(json.dumps(nf, ensure_ascii=False, indent=2), encoding="utf-8")
    logger.info(f"[{run_id}] Saved notion_fields.json: {notion_fields_path}")
    return nf

# ----------------------------
# Capability detection: can we pass "extra" and which params exist?
# ----------------------------
def _filter_kwargs_for_callable(fn, kwargs: Dict[str, Any]) -> Dict[str, Any]:
    """
    Keep only kwargs accepted by fn signature. If **kwargs exists, pass all.
    This makes Cell 08 resilient to minor signature changes.
    """
    try:
        sig = inspect.signature(fn)
        params = sig.parameters
        if any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values()):
            return kwargs
        allowed = set(params.keys())
        return {k: v for k, v in kwargs.items() if k in allowed}
    except Exception:
        return kwargs

def _extract_page_id(notion_result: Any) -> Optional[str]:
    if notion_result is None:
        return None
    if isinstance(notion_result, dict):
        return notion_result.get("page_id") or notion_result.get("id")
    return None

# ----------------------------
# Main: create Notion paper (INBOX)
# ----------------------------
def create_notion_paper_record(
    metadata: Dict[str, Any],
    pdf_path: Path,
    slide_path: Optional[Path] = None,
    openai_model: str = "gpt-4.1-mini",
    openai_temperature: float = 0.2,
) -> Dict[str, Any]:
    """
    Calls create_paper_inbox(name, authors_year, pdf_link, tags, status, pdf_status, dedup_key,
                            source_uid, run_id, slide1_url, extra)
    and returns {success, page_id, used_wrapper, payload_preview, error, dedup_key, source_uid}.
    """
    result = {
        "success": False,
        "page_id": None,
        "used_wrapper": "create_paper_inbox",
        "payload_preview": None,
        "error": None,
        "dedup_key": None,
        "source_uid": None,
    }

    create_fn = globals().get("create_paper_inbox")
    if not callable(create_fn):
        msg = "create_paper_inbox is not callable. Ensure 029 executed and exported correctly."
        logger.error(f"[{run_id}] {msg}")
        result["error"] = msg
        return result

    # identifiers
    title = (metadata.get("title") or "").strip() or (pdf_path.stem.strip() or "Untitled Paper")
    source_uid = _make_source_uid(metadata, pdf_path)
    dedup_key = _make_paper_dedup_key(source_uid)

    result["source_uid"] = source_uid
    result["dedup_key"] = dedup_key

    # artifact dir per paper
    pdf_run_dir = _get_pdf_run_dir(artifacts_path, dedup_key)

    # self-healing rich fields
    notion_fields = {}
    try:
        notion_fields = _get_or_generate_notion_fields(
            pdf_path=pdf_path,
            metadata=metadata,
            pdf_run_dir=pdf_run_dir,
            openai_model=OPENAI_MODEL,
            temperature=OPENAI_TEMPERATURE,
        )
    except Exception as e:
        logger.warning(f"[{run_id}] notion_fields generation failed; continuing. err={e}")
        notion_fields = {}

    # values to pass
    name = (notion_fields.get("name") or "").strip() or title
    authors_year = (notion_fields.get("authors_year") or "").strip() or _to_authors_year(metadata)
    tags = notion_fields.get("tags") if isinstance(notion_fields.get("tags"), list) else None
    if tags is not None:
        tags = [str(t).strip() for t in tags if str(t).strip()]
        if not tags:
            tags = None

    # status fields
    pdf_status = "LOCAL" if pdf_path.exists() else "NONE"
    slide1_url = None  # keep None unless you have an actual URL

    # extra payload (this is what later notebooks can use to backfill Notion properties)
    extra = {
        "pipeline": "031_pdf_inbox_processor",
        "ingested_at": datetime.today().date().isoformat(),
        "local_pdf_path": str(pdf_path),
        "local_slide_path": str(slide_path) if (slide_path and slide_path.exists()) else None,
        "pdf_run_dir": str(pdf_run_dir),
        "doi": (metadata.get("doi") or "").strip() or None,
        "arxiv_id": (metadata.get("arxiv_id") or "").strip() or None,
        "raw_title": title,
        "authors_list": metadata.get("authors") or [],
        "notion_fields": notion_fields or None,
    }
    extra = {k: v for k, v in extra.items() if v is not None}

    # build kwargs exactly for create_paper_inbox signature (but filter defensively)
    # pdf_link: keep None unless you have a real URL (local file path is stored in extra)
    pdf_link_value = None

    kwargs = dict(
        name=notion_fields.get("name") if isinstance(notion_fields.get("name"), str) and notion_fields.get("name").strip() else title,
        authors_year=notion_fields.get("authors_year") if isinstance(notion_fields.get("authors_year"), str) else authors_year,
        pdf_link=pdf_link_value,         # ✅ FIX: never reference undefined variable
        tags=tags,                       # list[str] or None
        status="INBOX",
        pdf_status=pdf_status,
        dedup_key=dedup_key,
        source_uid=source_uid,
        run_id=run_id,
        slide1_url=slide1_url,           # keep None unless URL
        extra=extra,                     # rich payload
    )
    

    kwargs = _filter_kwargs_for_callable(create_fn, kwargs)

    # preview for logs/debug
    result["payload_preview"] = {
        "name": (kwargs.get("name") or "")[:90],
        "authors_year": (kwargs.get("authors_year") or "")[:90],
        "status": kwargs.get("status"),
        "pdf_status": kwargs.get("pdf_status"),
        "dedup_key": kwargs.get("dedup_key"),
        "source_uid": kwargs.get("source_uid"),
        "tags": kwargs.get("tags"),
        "has_extra": "extra" in kwargs,
        "extra_keys": sorted(list((kwargs.get("extra") or {}).keys()))[:40] if isinstance(kwargs.get("extra"), dict) else None,
    }

    logger.info(f"[{run_id}] Creating Notion paper (INBOX): {name[:80]}...")
    logger.debug(f"[{run_id}] create_paper_inbox kwargs: {list(kwargs.keys())}")

    try:
        notion_result = create_fn(**kwargs)
        page_id = _extract_page_id(notion_result)

        if page_id:
            result["success"] = True
            result["page_id"] = page_id
            logger.info(f"[{run_id}] Notion record created: page_id={page_id}")
        else:
            msg = f"create_paper_inbox returned no page id (type={type(notion_result)})"
            logger.warning(f"[{run_id}] {msg}")
            result["error"] = msg

    except Exception as e:
        msg = f"Notion record creation failed via create_paper_inbox: {e}"
        logger.error(f"[{run_id}] {msg}")
        result["error"] = msg

    return result

logger.info(f"[{run_id}] Cell 08 ready: create_notion_paper_record() (create_paper_inbox-aligned + self-healing)")


2026-02-01 09:31:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Cell 08 ready: create_notion_paper_record() (create_paper_inbox-aligned + self-healing)


In [10]:
# ============================================================
# Cell 09: Single PDF processing pipeline (with optional Drive upload + Notion link update)
# ============================================================
import time
import shutil
from datetime import datetime
from typing import Dict, Any, Optional
from pathlib import Path

SKIP_SLIDE_IF_DUPLICATE = True
REQUIRE_OPENAI_FOR_FIELDS = False  # True: openai_client 無い場合 Notion作成を失敗扱い
ENABLE_DRIVE_UPLOAD = True         # Drive連携を使わないなら False

def _has_openai_client() -> bool:
    # openai_client は OpenAI SDKクライアントのインスタンスで callable ではないことが多い
    return "openai_client" in globals() and globals().get("openai_client") is not None

def _to_path(p) -> Optional[Path]:
    if p is None:
        return None
    if isinstance(p, Path):
        return p
    try:
        return Path(str(p))
    except Exception:
        return None

# ----------------------------
# Drive upload + Notion link update (optional)
# ----------------------------
def _drive_safe_title(s: str) -> str:
    s = (s or "").strip()
    s = s.replace("/", "_").replace(":", "").replace("\n", " ")
    return " ".join(s.split())

def _drive_title(base_title: str, suffix: str, ext: str) -> str:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    base = _drive_safe_title(base_title)[:120] or "paper"
    return f"{base}__{suffix}__{ts}.{ext}"

def _ensure_shareable_anyone_reader(drive_service, file_id: str):
    drive_service.permissions().create(
        fileId=file_id,
        body={"role": "reader", "type": "anyone"},
    ).execute()

def _drive_file_url(file_id: str) -> str:
    return f"https://drive.google.com/file/d/{file_id}/view"

def _upload_file_to_drive(
    drive_service,
    local_path: Path,
    drive_folder_id: str,
    title: str,
    mimetype: str,
) -> Dict[str, Any]:
    from googleapiclient.http import MediaFileUpload

    media = MediaFileUpload(str(local_path), mimetype=mimetype, resumable=True)
    created = drive_service.files().create(
        body={"name": title, "parents": [drive_folder_id]},
        media_body=media,
        fields="id,name",
    ).execute()

    file_id = created["id"]
    _ensure_shareable_anyone_reader(drive_service, file_id)
    return {"file_id": file_id, "name": created["name"], "url": _drive_file_url(file_id)}

def _upload_pdf_and_slide_to_drive(
    drive_service,
    pdf_path: Path,
    slide_path: Optional[Path],
    drive_folder_id: str,
    base_title: str,
) -> Dict[str, Any]:
    out = {"pdf": None, "slide": None}

    if pdf_path and pdf_path.exists():
        pdf_title = _drive_title(base_title, "paper", "pdf")
        out["pdf"] = _upload_file_to_drive(
            drive_service=drive_service,
            local_path=pdf_path,
            drive_folder_id=drive_folder_id,
            title=pdf_title,
            mimetype="application/pdf",
        )

    if slide_path and slide_path.exists():
        ext = slide_path.suffix.lstrip(".").lower() or "png"
        slide_title = _drive_title(base_title, "slide", ext)
        out["slide"] = _upload_file_to_drive(
            drive_service=drive_service,
            local_path=slide_path,
            drive_folder_id=drive_folder_id,
            title=slide_title,
            mimetype=f"image/{ext}",
        )

    return out

def _maybe_drive_upload_and_update_notion(
    *,
    status: str,
    page_id: Optional[str],
    pdf_path: Path,
    slide_path: Optional[Path],
    metadata: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Returns dict with keys: {ran, uploaded, pdf_url, slide_url, notion_update_result}
    Raises only on programmer errors; runtime errors should be caught by caller.
    """
    out = {
        "ran": False,
        "uploaded": None,
        "pdf_url": None,
        "slide_url": None,
        "notion_update_result": None,
    }

    if not ENABLE_DRIVE_UPLOAD:
        return out
    if status != "success" or not page_id:
        return out

    # prerequisites
    if "drive_service" not in globals() or globals().get("drive_service") is None:
        raise ValueError("drive_service not initialized (run Drive auth cell first).")
    if "DRIVE_FOLDER_ID" not in globals() or not globals().get("DRIVE_FOLDER_ID"):
        raise ValueError("DRIVE_FOLDER_ID is not set.")
    if not callable(globals().get("update_paper_links")):
        raise ValueError("update_paper_links is not callable (ensure 029 defines it and executed).")

    drive_service = globals()["drive_service"]
    drive_folder_id = globals()["DRIVE_FOLDER_ID"]

    base_title = (metadata.get("title") or pdf_path.stem).strip() or "paper"
    uploaded = _upload_pdf_and_slide_to_drive(
        drive_service=drive_service,
        pdf_path=pdf_path,
        slide_path=slide_path,
        drive_folder_id=drive_folder_id,
        base_title=base_title,
    )

    pdf_url = (uploaded.get("pdf") or {}).get("url")
    slide_url = (uploaded.get("slide") or {}).get("url")

    upd = update_paper_links(
        page_id=page_id,
        pdf_link=pdf_url,
        slide1_url=slide_url,
    )

    out["ran"] = True
    out["uploaded"] = uploaded
    out["pdf_url"] = pdf_url
    out["slide_url"] = slide_url
    out["notion_update_result"] = upd
    return out


# ----------------------------
# Main pipeline per PDF
# ----------------------------
def process_single_pdf(pdf_path: Path) -> Dict[str, Any]:
    start_time = time.time()

    result: Dict[str, Any] = {
        "pdf_name": pdf_path.name,
        "status": "failed",
        "page_id": None,
        "duplicate_info": None,
        "slide_path": None,
        "moved_to": None,
        "errors": [],
        "duration_seconds": 0.0,
        "timestamp": None,
        "dedup_key": None,
        "source_uid": None,
        "notion_payload_preview": None,
        "drive": None,  # 追加: Drive結果も追える
    }

    logger.info(f"[{run_id}] Processing PDF: {pdf_path.name}")
    logger.info(f"[{run_id}] {'='*60}")

    metadata: Optional[Dict[str, Any]] = None
    duplicate_check: Optional[Dict[str, Any]] = None
    slide_path: Optional[Path] = None

    try:
        # --- Stage 1: Metadata extraction ---
        logger.info(f"[{run_id}] Stage 1/5: Extracting metadata...")
        metadata = extract_pdf_metadata(pdf_path)

        if metadata.get("extraction_errors"):
            for err in metadata["extraction_errors"]:
                result["errors"].append(f"metadata: {err}")

        if not (metadata.get("title") or "").strip():
            msg = "No title resolved (unexpected)."
            logger.error(f"[{run_id}] {msg}")
            result["errors"].append(msg)
            return result

        # --- Stage 2: Deduplication check ---
        logger.info(f"[{run_id}] Stage 2/5: Checking for duplicates...")
        try:
            duplicate_check = check_for_duplicate(metadata)
            if duplicate_check.get("is_duplicate"):
                result["status"] = "duplicate"
                result["duplicate_info"] = duplicate_check
                logger.info(
                    f"[{run_id}] DUPLICATE FOUND type={duplicate_check.get('match_type')} page_id={duplicate_check.get('page_id')}"
                )
            else:
                logger.info(f"[{run_id}] No duplicate found")
            if duplicate_check.get("error"):
                result["errors"].append(f"dedup: {duplicate_check['error']}")
        except Exception as e:
            msg = f"dedup exception: {e}"
            logger.warning(f"[{run_id}] {msg}")
            result["errors"].append(msg)

        # --- Stage 3: Slide artifact generation ---
        if result["status"] == "duplicate" and SKIP_SLIDE_IF_DUPLICATE:
            logger.info(f"[{run_id}] Stage 3/5: Skipping slide generation (duplicate)")
        else:
            logger.info(f"[{run_id}] Stage 3/5: Generating slide artifact...")
            try:
                slide_path = _to_path(generate_slide_artifact(pdf_path, metadata))
                if slide_path and slide_path.exists():
                    result["slide_path"] = str(slide_path)
                    logger.info(f"[{run_id}] Slide OK: {slide_path.name}")
                else:
                    msg = "slide: generation returned None"
                    logger.warning(f"[{run_id}] {msg}")
                    result["errors"].append(msg)
            except Exception as e:
                msg = f"slide exception: {e}"
                logger.warning(f"[{run_id}] {msg}")
                result["errors"].append(msg)

        # --- Stage 4: Notion record creation ---
        if result["status"] == "duplicate":
            logger.info(f"[{run_id}] Stage 4/5: Skipping Notion creation (duplicate)")
        else:
            logger.info(f"[{run_id}] Stage 4/5: Creating Notion record...")

            if not _has_openai_client():
                warn = "openai_client is not defined; notion_fields may be empty (metadata-only record)."
                logger.warning(f"[{run_id}] {warn}")
                result["errors"].append(f"notion_fields: {warn}")
                if REQUIRE_OPENAI_FOR_FIELDS:
                    raise RuntimeError("Aborting Notion creation (openai_client missing) because REQUIRE_OPENAI_FOR_FIELDS=True")

            try:
                creation = create_notion_paper_record(
                    metadata=metadata,
                    pdf_path=pdf_path,
                    slide_path=slide_path,
                )

                result["dedup_key"] = creation.get("dedup_key")
                result["source_uid"] = creation.get("source_uid")
                result["notion_payload_preview"] = creation.get("payload_preview")

                if creation.get("success"):
                    result["status"] = "success"
                    result["page_id"] = creation.get("page_id")
                    logger.info(f"[{run_id}] Notion create OK page_id={result['page_id']}")
                else:
                    msg = f"notion: {creation.get('error') or 'unknown error'}"
                    logger.error(f"[{run_id}] {msg}")
                    result["errors"].append(msg)
                    result["status"] = "failed"
            except Exception as e:
                msg = f"notion exception: {e}"
                logger.error(f"[{run_id}] {msg}")
                result["errors"].append(msg)
                result["status"] = "failed"

        # --- Stage 4.5: Drive upload + Notion link update (optional) ---
        if metadata is None:
            metadata = {}
        try:
            drive_out = _maybe_drive_upload_and_update_notion(
                status=result["status"],
                page_id=result.get("page_id"),
                pdf_path=pdf_path,
                slide_path=slide_path,
                metadata=metadata,
            )
            result["drive"] = drive_out
            if drive_out.get("ran"):
                logger.info(f"[{run_id}] Drive upload & Notion link update OK")
        except Exception as e:
            msg = f"drive/notion-link-update exception: {e}"
            logger.warning(f"[{run_id}] {msg}")
            result["errors"].append(msg)

        # --- Stage 5: File movement ---
        logger.info(f"[{run_id}] Stage 5/5: Moving PDF file...")
        try:
            target_dir = processed_path if result["status"] in ("success", "duplicate") else failed_path
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            target_path = target_dir / f"{ts}_{pdf_path.name}"

            shutil.move(str(pdf_path), str(target_path))
            result["moved_to"] = str(target_path)
            logger.info(f"[{run_id}] Moved to: {target_path}")
        except Exception as e:
            msg = f"move exception: {e}"
            logger.error(f"[{run_id}] {msg}")
            result["errors"].append(msg)

    finally:
        result["duration_seconds"] = round(time.time() - start_time, 2)
        result["timestamp"] = datetime.now().isoformat()
        logger.info(
            f"[{run_id}] Done: {pdf_path.name} status={result['status']} duration={result['duration_seconds']}s errors={len(result['errors'])}"
        )
        logger.info(f"[{run_id}] {'='*60}")

    return result


logger.info(f"[{run_id}] Cell 09: process_single_pdf() ready (Drive upload optional; Drive auth must be run separately)")


2026-02-01 09:31:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Cell 09: process_single_pdf() ready (Drive upload optional; Drive auth must be run separately)


In [11]:
# ============================================================
# Cell 10 — Main processing loop and orchestration (defensive)
# ============================================================
# Overview:
#   Process all PDFs in inbox and aggregate run summary.
#
# Notes:
#   - Uses update_state if available (028-owned state)
#   - Avoids direct state[...] assignment

import time
from typing import List, Dict, Any
from datetime import datetime  # ✅ add for safety

logger.info(f"[{run_id}] Discovering PDF files in inbox...")
logger.info(f"[{run_id}] Inbox path: {inbox_path}")

def _safe_resolve(p):
    try:
        return p.expanduser().resolve()
    except Exception:
        # Fallback: keep as-is (still usable for reading/moving in most cases)
        return p

# Case-insensitive-ish discovery + deterministic ordering
candidates = list(inbox_path.glob("*.pdf")) + list(inbox_path.glob("*.PDF"))
pdf_files = sorted(
    {_safe_resolve(p) for p in candidates if p.is_file()},
    key=lambda p: p.name.lower()
)

logger.info(f"[{run_id}] Found {len(pdf_files)} PDF file(s) in inbox")

processing_results: List[Dict[str, Any]] = []

run_summary = {
    "run_id": run_id,
    "start_time": datetime.now().isoformat(),
    "end_time": None,
    "total_files": len(pdf_files),
    "processed": 0,
    "success": 0,
    "duplicates": 0,
    "failures": 0,
    "total_errors": 0,
    "total_duration_seconds": 0.0,
    "results": [],
}

logger.info(f"[{run_id}] Starting batch processing... total_files={run_summary['total_files']}")

batch_start_time = time.time()

for i, pdf_path in enumerate(pdf_files, 1):
    logger.info(f"[{run_id}] {'='*70}")
    logger.info(f"[{run_id}] Processing {i}/{len(pdf_files)}: {pdf_path.name}")
    logger.info(f"[{run_id}] {'='*70}")

    try:
        result = process_single_pdf(pdf_path)

        processing_results.append(result)
        run_summary["results"].append(result)
        run_summary["processed"] += 1

        st = result.get("status")
        if st == "success":
            run_summary["success"] += 1
            logger.info(f"[{run_id}] ✓ SUCCESS: {pdf_path.name}")
        elif st == "duplicate":
            run_summary["duplicates"] += 1
            logger.info(f"[{run_id}] ⊕ DUPLICATE: {pdf_path.name}")
        else:
            run_summary["failures"] += 1
            logger.warning(f"[{run_id}] ✗ FAILED: {pdf_path.name}")

        errs = result.get("errors") or []
        if errs:
            run_summary["total_errors"] += len(errs)
            for err in errs:
                logger.warning(f"[{run_id}]   - {err}")

        run_summary["total_duration_seconds"] += float(result.get("duration_seconds") or 0.0)

    except Exception as e:
        msg = f"Unexpected error processing {pdf_path.name}: {e}"
        logger.error(f"[{run_id}] {msg}")
        logger.exception(e)

        failed_result = {
            "pdf_name": pdf_path.name,
            "status": "failed",
            "page_id": None,
            "duplicate_info": None,
            "slide_path": None,
            "moved_to": None,
            "errors": [msg],
            "duration_seconds": 0.0,
            "timestamp": datetime.now().isoformat(),
        }

        processing_results.append(failed_result)
        run_summary["results"].append(failed_result)
        run_summary["processed"] += 1
        run_summary["failures"] += 1
        run_summary["total_errors"] += 1

    logger.info(
        f"[{run_id}] Progress: {i}/{len(pdf_files)} | "
        f"{run_summary['success']} success, {run_summary['duplicates']} dup, {run_summary['failures']} fail"
    )

batch_duration = round(time.time() - batch_start_time, 2)
run_summary["end_time"] = datetime.now().isoformat()
run_summary["batch_duration_seconds"] = batch_duration
run_summary["total_duration_seconds"] = round(run_summary["total_duration_seconds"], 2)
run_summary["avg_duration_per_file"] = round(
    (run_summary["total_duration_seconds"] / run_summary["processed"]) if run_summary["processed"] else 0.0, 2
)

# Persist via update_state (preferred), else skip
try:
    update_state("processing_results", processing_results)
    update_state("run_summary", run_summary)
except Exception:
    pass

logger.info(f"[{run_id}] {'='*70}")
logger.info(
    f"[{run_id}] BATCH COMPLETE | files={run_summary['total_files']} "
    f"success={run_summary['success']} dup={run_summary['duplicates']} fail={run_summary['failures']} "
    f"errors={run_summary['total_errors']} duration={batch_duration}s"
)
logger.info(f"[{run_id}] {'='*70}")

if run_summary["total_files"] == 0:
    logger.info(f"[{run_id}] Inbox empty; no action taken")

logger.info(f"[{run_id}] Cell 10: Done")


2026-02-01 09:31:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Discovering PDF files in inbox...
2026-02-01 09:31:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Inbox path: /Users/yuetoya/Desktop/researchOS100-private/notebooks/data/downloads
2026-02-01 09:31:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Found 3 PDF file(s) in inbox
2026-02-01 09:31:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Starting batch processing... total_files=3
2026-02-01 09:31:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] ======================================================================
2026-02-01 09:31:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Processing 1/3: 1-s2.0-S2667096826000029-main.pdf
2026-02-01 09:31:51 | INF

In [12]:
# ============================================================
# Cell 11 — Run summary generation and reporting (state-less safe)
# ============================================================
# Overview:
#   Generates human-readable summary report from run_summary (Cell 10).
#   Works even when `state` dict is not defined (028 style).
#
# Inputs / Outputs:
#   Inputs: run_summary + processing_results (preferred via get_state)
#   Outputs: formatted_summary dict, summary_text str (saved via update_state if available)

from typing import Dict, List, Any

logger.info(f"[{run_id}] Generating run summary report...")

# ----------------------------
# Helpers: state access (028-compatible)
# ----------------------------
def _get_state_value(key: str, default=None):
    """
    Preferred: get_state(key)
    Fallback: globals()[key] if present
    Fallback: state dict if exists (legacy)
    """
    # 1) 028-style getter
    if callable(globals().get("get_state")):
        try:
            v = get_state(key)
            if v is not None:
                return v
        except Exception:
            pass

    # 2) legacy "state" dict (if notebook still has it)
    st = globals().get("state", None)
    if isinstance(st, dict) and key in st:
        return st.get(key, default)

    # 3) globals fallback (Cell10 may have variables)
    if key in globals():
        return globals().get(key, default)

    return default


def _set_state_value(key: str, value):
    """
    Preferred: update_state(key, value)
    Fallback: globals()[key] assignment
    Fallback: state dict if exists (legacy)
    """
    if callable(globals().get("update_state")):
        try:
            update_state(key, value)
            return
        except Exception:
            pass

    st = globals().get("state", None)
    if isinstance(st, dict):
        st[key] = value
        return

    globals()[key] = value


# ----------------------------
# Retrieve run_summary + processing_results
# ----------------------------
run_summary = _get_state_value("run_summary")
processing_results = _get_state_value("processing_results", default=[])

if not isinstance(run_summary, dict):
    msg = "run_summary not found. Ensure Cell 10 executed and update_state('run_summary', run_summary) succeeded."
    logger.error(f"[{run_id}] {msg}")
    raise RuntimeError(msg)

if not isinstance(processing_results, list):
    processing_results = []

logger.debug(f"[{run_id}] Run summary keys: {list(run_summary.keys())}")
logger.debug(f"[{run_id}] Processing results count: {len(processing_results)}")

# ----------------------------
# Build structured summary
# ----------------------------
formatted_summary: Dict[str, Any] = {
    "run_metadata": {
        "run_id": run_summary.get("run_id"),
        "start_time": run_summary.get("start_time"),
        "end_time": run_summary.get("end_time"),
        "duration_seconds": run_summary.get("batch_duration_seconds", 0.0),
    },
    "statistics": {
        "total_files": run_summary.get("total_files", 0),
        "processed": run_summary.get("processed", 0),
        "success": run_summary.get("success", 0),
        "duplicates": run_summary.get("duplicates", 0),
        "failures": run_summary.get("failures", 0),
        "total_errors": run_summary.get("total_errors", 0),
        "avg_duration_per_file": run_summary.get("avg_duration_per_file", 0.0),
    },
    "success_details": [],
    "duplicate_details": [],
    "failure_details": [],
    "error_summary": [],
    "recommendations": [],
}

# ----------------------------
# Parse individual results
# ----------------------------
for result in processing_results:
    if not isinstance(result, dict):
        continue

    pdf_name = result.get("pdf_name", "(unknown)")
    status = result.get("status", "unknown")
    page_id = result.get("page_id")
    errors = result.get("errors", []) or []
    duration = float(result.get("duration_seconds") or 0.0)

    if status == "success":
        formatted_summary["success_details"].append(
            {"pdf_name": pdf_name, "page_id": page_id, "duration_seconds": duration}
        )

    elif status == "duplicate":
        dup_info = result.get("duplicate_info", {}) or {}
        formatted_summary["duplicate_details"].append(
            {
                "pdf_name": pdf_name,
                "match_type": dup_info.get("match_type", "unknown"),
                "existing_page_id": dup_info.get("page_id"),
                "existing_title": (dup_info.get("title", "(unknown)") or "(unknown)")[:50],
            }
        )

    else:
        formatted_summary["failure_details"].append(
            {"pdf_name": pdf_name, "errors": errors, "duration_seconds": duration}
        )
        for err in errors:
            formatted_summary["error_summary"].append(
                {"pdf_name": pdf_name, "error": str(err)[:200]}
            )

# ----------------------------
# Recommendations
# ----------------------------
stats = formatted_summary["statistics"]

if stats["failures"] > 0:
    formatted_summary["recommendations"].append(
        f"Review {stats['failures']} failed file(s) in failed/ directory"
    )
    formatted_summary["recommendations"].append(
        "Check logs for detailed error messages and PDF file integrity"
    )

if stats["duplicates"] > 0:
    formatted_summary["recommendations"].append(
        f"{stats['duplicates']} duplicate(s) found; verify deduplication logic if unexpected"
    )

if stats["total_files"] == 0:
    formatted_summary["recommendations"].append(
        "Inbox was empty; no action required (this is normal)"
    )

if stats["total_errors"] > stats["failures"]:
    formatted_summary["recommendations"].append(
        "Multiple errors per file detected; check for systemic issues (API rate limits, config errors)"
    )

# ----------------------------
# Plaintext summary (Slack-friendly)
# ----------------------------
summary_lines: List[str] = []
summary_lines.append("=" * 70)
summary_lines.append("PDF INBOX PROCESSING SUMMARY")
summary_lines.append("=" * 70)
summary_lines.append("")
summary_lines.append(f"Run ID: {formatted_summary['run_metadata']['run_id']}")
summary_lines.append(f"Start: {formatted_summary['run_metadata']['start_time']}")
summary_lines.append(f"End: {formatted_summary['run_metadata']['end_time']}")
summary_lines.append(f"Duration: {formatted_summary['run_metadata']['duration_seconds']:.2f}s")
summary_lines.append("")
summary_lines.append("STATISTICS")
summary_lines.append("-" * 40)
summary_lines.append(f"Total files: {stats['total_files']}")
summary_lines.append(f"Processed: {stats['processed']}")
summary_lines.append(f"Success: {stats['success']} ✓")
summary_lines.append(f"Duplicates: {stats['duplicates']} ⊕")
summary_lines.append(f"Failures: {stats['failures']} ✗")
summary_lines.append(f"Total errors: {stats['total_errors']}")
summary_lines.append(f"Avg time/file: {stats['avg_duration_per_file']:.2f}s")
summary_lines.append("")

if formatted_summary["success_details"]:
    summary_lines.append("SUCCESSFUL IMPORTS")
    summary_lines.append("-" * 40)
    for i, item in enumerate(formatted_summary["success_details"], 1):
        summary_lines.append(f"{i}. {item['pdf_name']}")
        summary_lines.append(f"   Page ID: {item['page_id']}")
        summary_lines.append(f"   Duration: {item['duration_seconds']:.2f}s")
    summary_lines.append("")

if formatted_summary["duplicate_details"]:
    summary_lines.append("DUPLICATES SKIPPED")
    summary_lines.append("-" * 40)
    for i, item in enumerate(formatted_summary["duplicate_details"], 1):
        summary_lines.append(f"{i}. {item['pdf_name']}")
        summary_lines.append(f"   Match type: {item['match_type']}")
        summary_lines.append(f"   Existing: {item['existing_title']}...")
        summary_lines.append(f"   Page ID: {item['existing_page_id']}")
    summary_lines.append("")

if formatted_summary["failure_details"]:
    summary_lines.append("FAILURES")
    summary_lines.append("-" * 40)
    for i, item in enumerate(formatted_summary["failure_details"], 1):
        summary_lines.append(f"{i}. {item['pdf_name']}")
        for err in item["errors"]:
            err_s = str(err)
            summary_lines.append(f"   ERROR: {err_s[:150]}..." if len(err_s) > 150 else f"   ERROR: {err_s}")
    summary_lines.append("")

if formatted_summary["recommendations"]:
    summary_lines.append("RECOMMENDATIONS")
    summary_lines.append("-" * 40)
    for i, rec in enumerate(formatted_summary["recommendations"], 1):
        summary_lines.append(f"{i}. {rec}")
    summary_lines.append("")

summary_lines.append("=" * 70)
summary_text = "\n".join(summary_lines)

# ----------------------------
# Persist outputs (028-style preferred)
# ----------------------------
_set_state_value("formatted_summary", formatted_summary)
_set_state_value("summary_text", summary_text)

logger.info(f"[{run_id}] Summary generation complete")
logger.info(f"[{run_id}] Run Summary:")
for line in summary_lines:
    logger.info(line)

logger.info(f"[{run_id}] Cell 11: Summary stored for Cell 12 (Slack output)")


2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Generating run summary report...
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Summary generation complete
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Run Summary:
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | ======================================================================
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | PDF INBOX PROCESSING SUMMARY
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | ======================================================================
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | 
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | Run ID: 3b30808f-8492-4d6d-a373-340713fc1a63
2026-02-01 09:33

In [13]:
# ============================================================
# Cell 12 — Daily summary output and Slack snippet (state-less safe)
# ============================================================
# Overview:
#   Generates final output artifacts for daily run reporting:
#   - Slack-formatted snippet (markdown-friendly plaintext)
#   - JSON summary file for automation/archival
#   Does NOT send Slack messages (external integration).
#
# Inputs:
#   - formatted_summary, summary_text (preferred via get_state from Cell 11)
#
# Outputs:
#   - slack_snippet (stored via update_state if available)
#   - summary_json_path (stored via update_state if available)
#   - JSON file written to artifacts/summaries/{run_id}_summary.json

import json
from pathlib import Path
from typing import Dict, Any

logger.info(f"[{run_id}] Preparing daily summary output...")

# ----------------------------
# Helpers: state access (028-compatible)
# ----------------------------
def _get_state_value(key: str, default=None):
    # 1) 028-style getter
    if callable(globals().get("get_state")):
        try:
            v = get_state(key)
            if v is not None:
                return v
        except Exception:
            pass

    # 2) legacy dict (if exists)
    st = globals().get("state", None)
    if isinstance(st, dict) and key in st:
        return st.get(key, default)

    # 3) globals fallback
    if key in globals():
        return globals().get(key, default)

    return default


def _set_state_value(key: str, value):
    if callable(globals().get("update_state")):
        try:
            update_state(key, value)
            return
        except Exception:
            pass

    st = globals().get("state", None)
    if isinstance(st, dict):
        st[key] = value
        return

    globals()[key] = value


# ----------------------------
# Retrieve inputs (from Cell 11)
# ----------------------------
formatted_summary = _get_state_value("formatted_summary")
summary_text = _get_state_value("summary_text", default="")

if not isinstance(formatted_summary, dict):
    msg = "formatted_summary not found. Ensure Cell 11 executed and stored outputs (update_state('formatted_summary', ...))."
    logger.error(f"[{run_id}] {msg}")
    raise RuntimeError(msg)

stats = formatted_summary.get("statistics", {}) or {}
run_meta = formatted_summary.get("run_metadata", {}) or {}

# guard defaults
total_files = int(stats.get("total_files", 0) or 0)
success = int(stats.get("success", 0) or 0)
duplicates = int(stats.get("duplicates", 0) or 0)
failures = int(stats.get("failures", 0) or 0)
total_errors = int(stats.get("total_errors", 0) or 0)
duration = float(run_meta.get("duration_seconds", 0.0) or 0.0)
avg_duration = float(stats.get("avg_duration_per_file", 0.0) or 0.0)

# ----------------------------
# Generate Slack-optimized snippet
# ----------------------------
logger.info(f"[{run_id}] Generating Slack-formatted snippet...")

slack_lines = []
slack_lines.append("📚 *PDF Inbox Processing Summary*")
slack_lines.append(f"`Run ID: {run_meta.get('run_id')}`")
slack_lines.append("")

slack_lines.append("*Statistics:*")
slack_lines.append(f"• Total files: {total_files}")
slack_lines.append(f"• ✅ Success: {success}")
slack_lines.append(f"• 🔄 Duplicates: {duplicates}")
slack_lines.append(f"• ❌ Failures: {failures}")
if total_errors > 0:
    slack_lines.append(f"• ⚠️ Total errors: {total_errors}")
slack_lines.append("")

slack_lines.append(f"⏱️ Duration: {duration:.1f}s (avg {avg_duration:.1f}s/file)")
slack_lines.append("")

# Successes (brief)
success_details = formatted_summary.get("success_details", []) or []
if success_details:
    slack_lines.append(f"*Successfully imported {len(success_details)} paper(s):*")
    for i, item in enumerate(success_details[:5], 1):
        slack_lines.append(f"{i}. `{item.get('pdf_name','(unknown)')}`")
    if len(success_details) > 5:
        slack_lines.append(f"   _(+{len(success_details) - 5} more)_")
    slack_lines.append("")

# Duplicates (brief)
dup_details = formatted_summary.get("duplicate_details", []) or []
if dup_details:
    slack_lines.append(f"*Skipped {len(dup_details)} duplicate(s):*")
    for i, item in enumerate(dup_details[:3], 1):
        slack_lines.append(
            f"{i}. `{item.get('pdf_name','(unknown)')}` (matched by {item.get('match_type','unknown')})"
        )
    if len(dup_details) > 3:
        slack_lines.append(f"   _(+{len(dup_details) - 3} more)_")
    slack_lines.append("")

# Failures (brief)
failure_details = formatted_summary.get("failure_details", []) or []
if failure_details:
    slack_lines.append(f"*⚠️ Failed {len(failure_details)} file(s):*")
    for i, item in enumerate(failure_details[:5], 1):
        slack_lines.append(f"{i}. `{item.get('pdf_name','(unknown)')}`")
        errs = item.get("errors", []) or []
        if errs:
            first_err = str(errs[0])
            slack_lines.append(f"   _{first_err[:100]}{'...' if len(first_err) > 100 else ''}_")
    if len(failure_details) > 5:
        slack_lines.append(f"   _(+{len(failure_details) - 5} more)_")
    slack_lines.append("")

# Recommendations
recs = formatted_summary.get("recommendations", []) or []
if recs:
    slack_lines.append("*Recommendations:*")
    for rec in recs[:3]:
        slack_lines.append(f"• {rec}")
    slack_lines.append("")

slack_lines.append("---")
slack_lines.append(f"_Completed at {run_meta.get('end_time')}_")

slack_snippet = "\n".join(slack_lines)

# ----------------------------
# Save JSON summary to artifacts
# ----------------------------
logger.info(f"[{run_id}] Saving JSON summary to artifacts...")

summaries_path = artifacts_path / "summaries"
summaries_path.mkdir(parents=True, exist_ok=True)

summary_json_path = summaries_path / f"{run_id}_summary.json"

json_summary: Dict[str, Any] = {
    "run_metadata": run_meta,
    "statistics": stats,
    "success_count": len(success_details),
    "duplicate_count": len(dup_details),
    "failure_count": len(failure_details),
    "success_details": success_details,
    "duplicate_details": dup_details,
    "failure_details": failure_details,
    "error_summary": formatted_summary.get("error_summary", []) or [],
    "recommendations": recs,
    "summary_text": summary_text,  # optional: keep full text too
}

summary_json_path.write_text(json.dumps(json_summary, indent=2, ensure_ascii=False), encoding="utf-8")

logger.info(f"[{run_id}] JSON summary saved: {summary_json_path.relative_to(artifacts_path)}")

# ----------------------------
# Persist outputs (028-style preferred)
# ----------------------------
_set_state_value("slack_snippet", slack_snippet)
_set_state_value("summary_json_path", str(summary_json_path))

# Expose numeric metrics for orchestrator aggregation (critical)
_set_state_value("pdfs_processed", int(len(success_details)))  # preferred source of truth
_set_state_value("run_stats", {
    "total_files": int(total_files),
    "success": int(success),
    "duplicates": int(duplicates),
    "failures": int(failures),
    "duration_seconds": float(duration),
})


# ----------------------------
# Log snippet to console
# ----------------------------
logger.info(f"[{run_id}] Slack-formatted snippet:")
logger.info("=" * 70)
for line in slack_lines:
    logger.info(line)
logger.info("=" * 70)

logger.info(f"[{run_id}] Cell 12: Ready for external reporting (Slack/email/etc.)")
logger.info(f"[{run_id}] 031_pdf_inbox_processor: Complete")


2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Preparing daily summary output...
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Generating Slack-formatted snippet...
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Saving JSON summary to artifacts...
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] JSON summary saved: summaries/3b30808f-8492-4d6d-a373-340713fc1a63_summary.json
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | [3b30808f-8492-4d6d-a373-340713fc1a63] Slack-formatted snippet:
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | ======================================================================
2026-02-01 09:33:51 | INFO     | 3b30808f-8492-4d6d-a373-340713fc1a63 | 📚 *PDF Inbox Proc